# <font color="#418FDE" size="6.5" uppercase>**B: Intermediate Segmentation & Detection Experiments**</font>
----

> Last update: 20241008

By the end of this lecture, you will be able to:

* Develop transferred-learned & ne-tuned models for various segmentation tasks
* Apply pre-trained models for various segmentation tasks


## **1. VOC Experiment**

In [ ]:
#@title Install Necessary Libraries & Restart the Session

# Installs a specific version (7.7.1) of the 'ipywidgets' library.
# This library is commonly used in Jupyter notebooks to create interactive widgets for displaying objects like sliders, buttons, & more.
# Link: https://ipywidgets.readthedocs.io/en/stable/
!pip install ipywidgets==7.7.1

# Importing the time module, which provides functions for handling time-related tasks.
# We'll use this later to create a pause (delay) in the script.
import time

# Importing 'clear_output' from the IPython.display module.
# 'clear_output' is a function that clears the output of the cell when it is run.
# This is useful to prevent clutter & display a cleaner output after installing packages.
# Link: https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.clear_output
from IPython.display import clear_output

# Calling the 'clear_output()' function to remove the installation output (if any).
# This ensures that once the installation is done, the cell output is cleared for a more user-friendly experience.
clear_output()

# This print statement confirms that the necessary libraries have been installed.
print("Necessary Libraries are Installed. Restarting the session!")

# Pausing the execution of the program for 1 second to ensure the print statement is visible before the session exits.
time.sleep(1)

# Importing 'os', which is the standard library module for interacting with the operating system.
# Here, it is used to exit the current session (Google Colab or Jupyter notebook).
import os

# This command forcefully exits the Python interpreter.
# In Google Colab, it will prompt the environment to restart to apply new installations (especially after installing ipywidgets).
# 'os._exit(00)' causes an immediate program termination with the status code 0 (which means success).
# The purpose here is to restart the notebook's kernel after installing ipywidgets, ensuring that the new packages are loaded properly.
os._exit(00)

In [ ]:
#@title Import Libraries

# PyTorch is a popular open-source machine learning library used for deep learning tasks.
# It provides features such as automatic differentiation, tensor computation, & more.
# Link: https://pytorch.org/
import torch

# 'optim' is a sub-module in PyTorch that contains various optimization algorithms like SGD, Adam, etc.
# These optimizers are crucial for training neural networks by updating the model weights based on the gradients.
# Link: https://pytorch.org/docs/stable/optim.html
import torch.optim as optim

# 'nn' is the neural network module in PyTorch.
# It contains pre-built layers & utilities for building deep learning models like convolutional layers, fully connected layers, etc.
# Link: https://pytorch.org/docs/stable/nn.html
import torch.nn as nn

# 'torchvision' is a package that provides image processing utilities, pre-trained models, & datasets for computer vision tasks.
# This is often used in object detection, image classification, segmentation, etc.
# Link: https://pytorch.org/vision/stable/index.html
import torchvision

# 'transforms' is a submodule in torchvision used for common image transformations, like resizing, normalization, converting to tensors, etc.
# These transformations are important for pre-processing images before feeding them into models.
# Link: https://pytorch.org/vision/stable/transforms.html
import torchvision.transforms as T

# 'matplotlib.pyplot' is a plotting library that is widely used for creating static, animated, & interactive visualizations in Python.
# It will be useful for visualizing images, results, & metrics during model training & evaluation.
# Link: https://matplotlib.org/stable/tutorials/introductory/pyplot.html
import matplotlib.pyplot as plt

# 'mpatches' is a sub-module in matplotlib used for drawing various shapes, like rectangles, polygons, circles, etc.
# In object detection or segmentation, it can be used to overlay bounding boxes or highlight regions on an image.
import matplotlib.patches as mpatches

# 'numpy' is a foundational package for numerical computing in Python.
# It supports large multi-dimensional arrays & matrices & provides a wide variety of mathematical operations.
# Link: https://numpy.org/doc/stable/
import numpy as np

# 'tqdm' is a library for showing progress bars in loops, which is particularly useful in training deep learning models where iteration progress is tracked.
# 'tqdm.notebook' version is specifically designed for displaying progress bars in Jupyter or Colab notebooks.
# Link: https://tqdm.github.io/
from tqdm.notebook import tqdm

# Importing 'DeepLabHead' which is the head (final layers) of the DeepLabV3 model, used for semantic segmentation tasks.
# DeepLabV3 is a state-of-the-art model architecture for semantic segmentation.
# Link: https://arxiv.org/abs/1706.05587 (Original DeepLabV3 Paper)
from torchvision.models.segmentation.deeplabv3 import DeepLabHead

# Importing pre-trained weights for the DeepLabV3 model with a ResNet-101 backbone.
# This allows for fine-tuning or using a pre-trained model directly for segmentation tasks.
# Link: https://pytorch.org/vision/stable/models.html (See DeepLabV3 with ResNet101)
from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights

# 'jaccard_score' is imported from 'sklearn.metrics'.
# This function is used to compute the Jaccard Index (or Intersection over Union, IoU), which is a key metric in image segmentation for evaluating model performance.
# Link: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.jaccard_score.html
from sklearn.metrics import jaccard_score

# 'defaultdict' is a dictionary subclass from the 'collections' module.
# It is used to provide a default value for a dictionary when the key does not exist.
# This can be useful in many machine learning tasks, like keeping track of metrics during training.
# Link: https://docs.python.org/3/library/collections.html#collections.defaultdict
from collections import defaultdict

In [ ]:
#@title Define Transformation Functions for the Segmentation Experiment

# This function applies transformations to the input segmentation mask, which typically consists of pixel-wise labels (e.g., 0 for background, 1 for object).
# The transformation includes resizing & handling ignored labels (255).
def transform_mask(mask):
    # Resize the mask to 256x256 pixels using nearest neighbor interpolation, which is appropriate for segmentation masks.
    # Nearest neighbor interpolation ensures that the discrete label values in the mask remain unchanged (unlike bilinear interpolation, which could introduce blending).
    # Link: https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.Resize
    mask = T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST)(mask)

    # Optional augmentations (commented out here) like random horizontal flips & random rotations are common in training to improve generalization.
    # They are useful for increasing the diversity of training data. In this case, they are commented out, meaning the mask will not undergo these transformations.
    # Uncommenting these would apply the transformation & help improve the model's robustness to different orientations.
    #mask = T.RandomHorizontalFlip()(mask)  # Randomly flips the mask horizontally (50% chance).
    #mask = T.RandomRotation(10, interpolation=T.InterpolationMode.NEAREST)(mask)  # Randomly rotates the mask by up to 10 degrees.

    # Convert the mask into a NumPy array for easier manipulation of its pixel values.
    mask = np.array(mask)

    # Ensure that the pixels with the value 255, which are commonly used as the "ignore index" in segmentation (i.e., areas where the loss function should ignore during training), are preserved.
    mask[mask == 255] = 255  # Keeps ignore index intact.

    # Convert the mask back into a PyTorch tensor, ensuring it's of type `long` since it represents class labels (integers, not floating point).
    # Link: https://pytorch.org/docs/stable/tensors.html
    return torch.tensor(mask, dtype=torch.long)


# This function is similar to `transform_mask`, but it does not apply data augmentations (e.g., no horizontal flip or rotation).
# This could be used for evaluation or testing, where you generally do not want to apply augmentations, only resize the mask.
def transform_mask_no_aug(mask):
    # Resize the mask to 256x256 using nearest neighbor interpolation to keep pixel labels intact.
    mask = T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST)(mask)

    # Convert the mask into a NumPy array.
    mask = np.array(mask)

    # Retain the "ignore index" (255) for areas where loss should be ignored during training.
    mask[mask == 255] = 255  # Keeps ignore index intact.

    # Convert the mask to a PyTorch tensor of type `long` (appropriate for class labels).
    return torch.tensor(mask, dtype=torch.long)


# This is a composition of several transformations applied to input images (not masks) in the segmentation task.
# These transformations include resizing, data augmentations, & normalizations.
transform_image = T.Compose([
    # Resize the input image to 256x256 pixels. This ensures consistency in input dimensions.
    # Link: https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.Resize
    T.Resize((256, 256)),

    # Optional data augmentations (commented out). These would randomly flip or rotate images, helping the model generalize better.
    #mask = T.RandomHorizontalFlip(),  # Randomly flip the image horizontally (50% chance).
    #mask = T.RandomRotation(10),  # Randomly rotate the image by up to 10 degrees.

    # Apply random color jitter to the image, altering brightness, contrast, saturation, & hue.
    # This type of augmentation helps the model generalize better by making it more invariant to changes in lighting or color conditions in the input images.
    # Link: https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.ColorJitter
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),

    # Convert the image into a PyTorch tensor & scale pixel values to the range [0, 1] (from the original [0, 255]).
    # Link: https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.ToTensor
    T.ToTensor(),

    # Normalize the image using pre-defined mean & standard deviation values.
    # These values (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) are the normalization constants used for models pre-trained on the ImageNet dataset.
    # Normalization helps with faster & more stable training.
    # Link: https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.Normalize
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
#@title Essential Functions
def initialize_weights(m):
    """
    This function initializes the weights of a model's layers.
    Specifically, it applies Kaiming (He) Normal initialization to convolutional layers and
    ensures biases are properly initialized.

    Args:
        m: A PyTorch layer that will be checked & initialized if it is a Conv2d layer.
    """

    # Check if the layer 'm' is an instance of nn.Conv2d, which is a 2D convolutional layer commonly used in CNNs.
    if isinstance(m, nn.Conv2d):

        # Apply Kaiming (He) Normal initialization to the weights of the Conv2d layer.
        # Kaiming initialization helps preserve the gradient's variance during backpropagation, especially in layers with ReLU activation functions.
        # This ensures more stable & efficient training, especially for deep neural networks.
        # Link: https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_normal_
        nn.init.kaiming_normal_(m.weight)

        # Check if the layer has a bias term. If it does, initialize the biases to 0.
        # Initializing biases to zero is a common practice to avoid adding any initial bias to the network’s predictions.
        # Link: https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.constant_
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

def calculate_class_weights(dataset, num_classes=21):
    """
    This function calculates class weights for segmentation tasks, based on the frequency of each class in the dataset.
    Class weights are often used to handle imbalanced datasets by penalizing underrepresented classes more during loss computation.

    Args:
        dataset: The training dataset, which contains tuples of (image, mask). The mask is a label mask where each pixel belongs to a class.
        num_classes: The number of classes in the segmentation task. Defaults to 21 (e.g., for PASCAL VOC segmentation dataset).

    Returns:
        A PyTorch tensor containing class weights. Each class gets a weight inversely proportional to its occurrence in the dataset.
    """
    # 'class_counts' will store the number of pixels for each class across all masks in the dataset.
    class_counts = defaultdict(int)

    # Loop through the dataset, which contains image & mask pairs.
    # tqdm is used to show a progress bar, which helps track the calculation process over large datasets.
    for _, mask in tqdm(dataset, desc="Calculating class weights"):
        # Convert the mask from a PyTorch tensor to a NumPy array for easier manipulation & pixel counting.
        mask = mask.numpy()

        # Count the number of pixels for each class (cls) in the mask & update 'class_counts'.
        # 'mask == cls' creates a boolean mask for the current class, & np.sum counts how many pixels belong to that class.
        for cls in range(num_classes):
            class_counts[cls] += np.sum(mask == cls)

    # Calculate the total number of pixels across all masks.
    total_pixels = sum(class_counts.values())

    # Create an empty list to store the weights for each class.
    class_weights = []

    # Calculate the weight for each class based on its occurrence in the dataset.
    for cls in range(num_classes):
        if class_counts[cls] > 0:
            # If the class has pixels, its weight is inversely proportional to the frequency of its occurrence.
            # The more frequent the class, the lower the weight, & vice versa.
            class_weights.append(total_pixels / (num_classes * class_counts[cls]))
        else:
            # If a class is not present in the dataset (no pixels), give it a default weight of 1.0 to avoid division by zero.
            class_weights.append(1.0)

    # Convert the list of class weights to a PyTorch tensor of type float32.
    # The '.to(device)' part ensures that the tensor is moved to the appropriate device (CPU or GPU).
    # 'device' is assumed to be predefined somewhere in the notebook.
    return torch.tensor(class_weights, dtype=torch.float32).to(device)

def calculate_mIoU(preds, targets, num_classes=21):
    """
    Calculates the mean Intersection over Union (mIoU) between predicted & ground truth segmentation masks.
    mIoU is the average IoU across all classes, which measures the overlap between predicted & true masks.

    Args:
        preds: The predicted segmentation masks. This is a tensor where each pixel has been assigned a predicted class.
        targets: The ground truth segmentation masks. This is a tensor where each pixel is assigned the correct class.
        num_classes: The number of classes in the segmentation task. Default is 21 (e.g., PASCAL VOC dataset).

    Returns:
        The mean Intersection over Union (mIoU) score as a float.
    """

    # Move predictions & targets from the GPU (if applicable) to the CPU & flatten them into 1D arrays.
    # 'flatten()' converts multi-dimensional arrays into 1D arrays, which simplifies pixel-wise comparisons.
    preds = preds.cpu().numpy().flatten()
    targets = targets.cpu().numpy().flatten()

    # Remove pixels labeled with 255 in the targets. In segmentation tasks, 255 typically represents an "ignore" label.
    # These are pixels where the model should not be penalized for incorrect predictions.
    # Both predictions & targets are filtered to remove corresponding ignored pixels.
    preds = preds[targets != 255]
    targets = targets[targets != 255]

    # If, after removing ignored pixels, the remaining target array is empty, return an mIoU of 0.0 to avoid errors.
    if len(targets) == 0:
        return 0.0

    # Calculate the Jaccard Index (IoU) using the sklearn's jaccard_score function.
    # 'average='macro'' computes the IoU for each class & then takes the average, treating all classes equally.
    # 'zero_division=0' ensures that the function handles divisions by zero gracefully.
    # 'labels=list(range(num_classes))' ensures that only the specified number of classes are used in the calculation.
    # Link: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.jaccard_score.html
    return jaccard_score(targets, preds, average='macro', zero_division=0, labels=list(range(num_classes)))

def training(model,
             train_loader,
             num_epochs,
             optimizer,
             criterion,
             scheduler=None,
             device="cuda",
             best_loss=float('inf'),
             lr_min=1e-5,
             validation_loader=None,
             epoch_patience=None,
             lr_patience=None,
             model_name=None,
             scaler=None):
    """
    Training function for semantic segmentation models using PyTorch.
    This function supports features such as mixed precision training, learning rate scheduling,
    early stopping, & model checkpointing.

    Args:
        model: The PyTorch model to be trained.
        train_loader: DataLoader for the training dataset.
        num_epochs: The number of epochs to train the model for.
        optimizer: The optimizer used to update model parameters (e.g., Adam, SGD).
        criterion: The loss function (e.g., CrossEntropyLoss).
        scheduler: Optional learning rate scheduler (e.g., ReduceLROnPlateau or StepLR).
        device: The device to run training on (default is "cuda" for GPU).
        best_loss: Initial best loss for tracking improvements in validation loss.
        lr_min: The minimum learning rate for LR scheduler (to prevent LR from becoming too small).
        validation_loader: DataLoader for the validation dataset (used if provided).
        epoch_patience: Number of epochs to wait for improvement in validation loss before early stopping.
        lr_patience: Number of epochs to wait before reducing the learning rate.
        model_name: Name of the model for saving checkpoint files.
        scaler: Gradient scaler for mixed precision training (if using amp for training).

    Returns:
        model: The trained model.
        history: A dictionary containing training loss, validation loss, & validation mIoU history.
    """

    # Early stopping counters
    if epoch_patience:
        epoch_patience_counter = 0

    if lr_patience:
        lr_patience_counter = 0

    # Lists to keep track of the training & validation losses & validation mIoU scores
    train_loss = []
    validation_loss = []
    validation_mIoU = []

    # Main training loop for the specified number of epochs
    for epoch in range(num_epochs):
        running_loss = 0.0
        model.train()  # Set the model to training mode

        # Display a progress bar using tqdm for monitoring each epoch's progress
        progress_bar = tqdm(train_loader,
                            desc=f"Epoch {epoch + 1}/{num_epochs}, LR: {optimizer.param_groups[0]['lr']:.6f}",
                            unit="batch",
                            leave=True,
                            dynamic_ncols=True)

        # Iterate through batches of training data
        for i, (images, masks) in enumerate(progress_bar, start=1):
            # Move images & masks to the specified device (GPU or CPU)
            images = images.to(device).float()
            masks = masks.to(device).long()

            optimizer.zero_grad()  # Clear gradients from the previous iteration

            # Mixed precision training block (if scaler is provided)
            if scaler:
                with torch.amp.autocast(device_type='cuda'):
                    outputs = model(images)['out']  # Forward pass
                    loss = criterion(outputs, masks)  # Calculate loss
                scaler.scale(loss).backward()  # Backward pass (gradient computation)
                scaler.unscale_(optimizer)  # Unscaling gradients before clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping to prevent exploding gradients
                scaler.step(optimizer)  # Step the optimizer with scaled gradients
                scaler.update()  # Update the scaler for the next iteration
            else:
                outputs = model(images)['out']  # Forward pass
                loss = criterion(outputs, masks)  # Calculate loss
                loss.backward()  # Backward pass (gradient computation)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
                optimizer.step()  # Update model parameters

            running_loss += loss.item()  # Accumulate the loss
            ave_loss = running_loss / i  # Average loss for the current epoch
            progress_bar.set_postfix(loss=f"{ave_loss:.5f}")  # Update progress bar with the current average loss

        train_loss.append(ave_loss)  # Store the average training loss for this epoch

        # If a validation loader is provided, evaluate the model on the validation set
        if validation_loader:
            val_loss, val_mIoU = evaluate_model(validation_loader, model)  # Evaluate model (assumed function exists)
            validation_loss.append(val_loss)  # Store validation loss
            validation_mIoU.append(val_mIoU)  # Store validation mIoU score
            progress_bar.set_postfix(loss=f"| Ave: {ave_loss:.5f} | Val: {val_loss:.5f} | mIoU: {val_mIoU:.4f}")  # Update progress bar

            # If a learning rate scheduler is provided, update it based on validation loss
            if scheduler:
                if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(val_loss)  # For ReduceLROnPlateau, we pass the validation loss
                else:
                    scheduler.step()  # Step the scheduler for other types like StepLR

            # Early Stopping logic: stop training if validation loss doesn't improve for `epoch_patience` epochs
            if epoch_patience:
                if val_loss < best_loss:  # If current validation loss is better, update best_loss & reset patience counter
                    best_loss = val_loss
                    epoch_patience_counter = 0
                    # Save the best model's state to a file
                    name = 'model.pth' if not model_name else f"{model_name}.pth"
                    torch.save(model.state_dict(), name)
                else:
                    epoch_patience_counter += 1  # Increment patience counter if no improvement

                # If no improvement for `epoch_patience` epochs, stop training early
                if epoch_patience_counter >= epoch_patience:
                    print(f"Early stopping at epoch {epoch + 1}")
                    break

        # Store training & validation history in a dictionary for later use
        history = {
            'training_loss': train_loss,
            'validation_loss': validation_loss,
            'validation_mIoU': validation_mIoU
        }

    # Save the final model at the end of training
    name = 'model_final.pth' if not model_name else f"{model_name}_final.pth"
    torch.save(model.state_dict(), name)

    return model, history  # Return the trained model & the training history

def evaluate_model(validation_loader, model):
    """
    Evaluates the model's performance on a validation dataset by calculating the validation loss
    & the mean Intersection over Union (mIoU) score.

    Args:
        validation_loader: DataLoader for the validation dataset.
        model: The PyTorch model being evaluated.

    Returns:
        avg_loss: The average loss over the entire validation dataset.
        avg_mIoU: The average mIoU score over the entire validation dataset.
    """

    # Set the model to evaluation mode, which disables certain behaviors such as dropout & batch normalization updates.
    model.eval()

    # Initialize variables to accumulate validation loss & mIoU score.
    validation_loss = 0.0
    total_mIoU = 0.0

    # Disable gradient calculation, which reduces memory usage & speeds up the evaluation process.
    with torch.no_grad():
        # Iterate over batches in the validation DataLoader.
        for images, masks in validation_loader:
            # Move the images & masks to the device (either GPU or CPU) & ensure correct data types.
            images, masks = images.to(device).float(), masks.to(device).long()

            # Remove extra dimensions from masks (if masks have a singleton dimension).
            masks = masks.squeeze(1)  # Assuming masks are shaped (B, 1, H, W), squeeze reduces it to (B, H, W).

            # Perform a forward pass through the model to obtain the predicted output.
            outputs = model(images)['out']

            # Calculate the loss between predicted outputs & ground truth masks using the criterion (loss function).
            loss = criterion(outputs, masks)

            # Accumulate the validation loss.
            validation_loss += loss.item()

            # Get the predicted class for each pixel by taking the class with the highest score (argmax over channels).
            preds = torch.argmax(outputs, dim=1)

            # Calculate mIoU for the current batch & accumulate it across batches.
            total_mIoU += calculate_mIoU(preds, masks)

    # Compute the average loss across all batches.
    avg_loss = validation_loss / len(validation_loader)

    # Compute the average mIoU across all batches.
    avg_mIoU = total_mIoU / len(validation_loader)

    return avg_loss, avg_mIoU


In [ ]:
#@title Data Preparation

# Custom Dataset to apply the same transformations to both image & mask
class VOCSegmentationDataset(torchvision.datasets.VOCSegmentation):
    """
    This custom dataset class inherits from torchvision's VOCSegmentation class.
    It allows for applying separate transformations to images & masks (segmentation labels),
    which is necessary in segmentation tasks where different transformations may be needed for images & masks.

    Args:
        transform_image (callable, optional): Transformation to apply to the image.
        transform_mask (callable, optional): Transformation to apply to the mask (segmentation label).
    """

    def __init__(self, *args, **kwargs):
        # Extract transformation functions for image & mask from the provided arguments.
        self.transform_image = kwargs.pop('transform', None)  # Transformation for the image
        self.transform_mask = kwargs.pop('target_transform', None)  # Transformation for the mask
        self.transforms = kwargs.pop('transforms', None)  # Placeholder for joint transforms (if any)

        # Call the superclass initializer without transformations (handled internally in __getitem__).
        super(VOCSegmentationDataset, self).__init__(*args, transform=None, target_transform=None, **kwargs)

    def __getitem__(self, index):
        """
        Override the __getitem__ method to load & apply transformations separately to the image & mask.

        Args:
            index (int): Index of the image-mask pair to retrieve.

        Returns:
            tuple: Transformed image & mask (image, mask).
        """
        # Retrieve the image & mask from the parent class method.
        image, mask = super(VOCSegmentationDataset, self).__getitem__(index)

        # Apply the image transformation (e.g., resizing, normalization) if defined.
        if self.transform_image:
            image = self.transform_image(image)

        # Apply the mask transformation (e.g., resizing, label adjustments) if defined.
        if self.transform_mask:
            mask = self.transform_mask(mask)

        # Return the transformed image & mask.
        return image, mask

# Data preparation for training & validation

# Define batch sizes for training & validation datasets.
training_batch_size = 8
validation_batch_size = 8

# Create a dataset instance for the training set with data augmentations applied to both images & masks.
dataset_train_aug = VOCSegmentationDataset(
    root="./",               # Root directory for the dataset.
    year="2008",             # Specify the year of the VOC dataset (e.g., VOC 2008).
    image_set="train",       # Use the training set for this dataset.
    download=True,           # Download the dataset if it's not already available.
    transform=transform_image,  # Apply the defined image transformations (e.g., resizing, normalization).
    target_transform=transform_mask  # Apply the defined mask transformations (e.g., resizing, handling ignore labels).
)

# Create another dataset instance without augmentation for calculating class weights.
# This is typically done to ensure the weights reflect the true class distribution without transformations affecting the masks.
dataset_class_weights = VOCSegmentationDataset(
    root="./",               # Root directory for the dataset.
    year="2008",             # Use the same VOC year.
    image_set="train",       # Use the training set.
    download=False,          # No need to download again since it's already downloaded.
    transform=None,          # No transformations for the images.
    target_transform=transform_mask_no_aug  # Apply mask transformation, but without augmentations (used for class weight calculations).
)

# Split the dataset into training (80%) & validation (20%) sets.
train_size = int(0.8 * len(dataset_train_aug))  # Calculate the training set size (80% of total dataset).
validation_size = len(dataset_train_aug) - train_size  # The remaining 20% is used for validation.
train_dataset, validation_dataset = torch.utils.data.random_split(
    dataset_train_aug, [train_size, validation_size])  # Split the dataset into training & validation sets.

# Create DataLoader for the training dataset.
train_loader = torch.utils.data.DataLoader(
    train_dataset,           # The training dataset split.
    batch_size=training_batch_size,  # Batch size for training.
    shuffle=True,            # Shuffle the data to ensure randomization for each epoch.
    num_workers=4,           # Number of worker processes to load data in parallel (speeds up loading).
    pin_memory=True          # Pin memory to speed up data transfer to the GPU.
)

# Create DataLoader for the validation dataset.
validation_loader = torch.utils.data.DataLoader(
    validation_dataset,      # The validation dataset split.
    batch_size=validation_batch_size,  # Batch size for validation.
    shuffle=False,           # No need to shuffle validation data (evaluation typically doesn't require this).
    num_workers=4,           # Number of worker processes for data loading.
    pin_memory=True          # Pin memory for efficient data transfer to the GPU.
)

In [ ]:
#@title Download & Prepare the Model for Training
# Model setup

# Select the device to train on: GPU (cuda) if available, otherwise fall back to CPU.
# GPU training is typically much faster, especially for deep learning models like DeepLabV3.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set the number of classes for the segmentation task.
# VOCSegmentation dataset has 21 classes (20 foreground object classes + 1 background class).
num_classes = 21  # VOCSegmentation has 21 classes

# Load a pre-trained DeepLabV3 model with a ResNet-101 backbone.
# 'weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1' loads weights pre-trained on COCO with VOC labels.
# Pre-trained models are used for transfer learning, allowing the model to learn faster & potentially achieve better performance.
# Link: https://pytorch.org/vision/stable/models.html#torchvision.models.segmentation.deeplabv3_resnet101
model = torchvision.models.segmentation.deeplabv3_resnet101(weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1)

# Replace the classifier head of the model (DeepLabHead) to match the number of classes (21 for VOC dataset).
# The DeepLabHead takes as input 2048 feature channels (from ResNet) & outputs the desired number of classes.
# This is a typical step in transfer learning when fine-tuning a pre-trained model for a specific dataset.
# Link: https://pytorch.org/vision/stable/_modules/torchvision/models/segmentation/deeplabv3.html#DeepLabV3
model.classifier = DeepLabHead(2048, num_classes)

# Initialize the weights of the new classifier head.
# This is important because the head was just added & hasn't been pre-trained, so its weights must be initialized.
model.classifier.apply(initialize_weights)

# Move the model to the selected device (GPU or CPU).
model.to(device)

# Calculate class weights based on the training dataset.
# Class weights are used in the loss function to handle class imbalance (e.g., rare classes get higher weights).
# The function 'calculate_class_weights' computes these weights based on the pixel distribution of each class in the dataset.
class_weights = calculate_class_weights(dataset_class_weights, num_classes=num_classes)

# Define the loss function for the segmentation task.
# CrossEntropyLoss is the standard loss for multi-class classification tasks, including segmentation.
# The 'weight' argument adjusts the loss to account for class imbalance, & 'ignore_index=255' tells the loss function
# to ignore pixels labeled with 255 (usually used as an ignore/mask region in VOCSegmentation).
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)

# Define the optimizer for transfer learning.
# SGD (Stochastic Gradient Descent) is commonly used, with momentum to accelerate convergence & weight_decay (L2 regularization) to prevent overfitting.
# The learning rate (lr=1e-3) is set higher for transfer learning since the base model's layers need to be fine-tuned.
optimizer_transfer_learning = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

# Define the learning rate scheduler for transfer learning.
# ReduceLROnPlateau reduces the learning rate when a metric (in this case, validation loss) stops improving.
# 'factor=0.5' halves the learning rate, 'patience=5' means it waits for 5 epochs before reducing the LR if no improvement is seen.
scheduler_transfer_learning = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_transfer_learning, mode='min', factor=0.5, patience=5, verbose=True)

# Define the optimizer for fine-tuning.
# In fine-tuning, a smaller learning rate (lr=1e-4) is used because the model's weights are already close to a good solution.
optimizer_fine_tuning = optim.SGD(model.parameters(), lr=1e-4, momentum=0.9, weight_decay=1e-4)

# Define the learning rate scheduler for fine-tuning.
# Same as the transfer learning scheduler but with 'patience=3', meaning it reduces the learning rate more quickly if no improvement.
scheduler_fine_tuning = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fine_tuning, mode='min', factor=0.5, patience=3, verbose=True)

# Initialize GradScaler for mixed precision training (optional but recommended for modern GPUs).
# Mixed precision reduces memory usage & speeds up training by using 16-bit floating-point precision where possible.
# The scaler dynamically adjusts the scaling of gradients to avoid underflow or overflow issues during backpropagation.
# Link: https://pytorch.org/docs/stable/amp.html#torch.cuda.amp.GradScaler
scaler = torch.amp.GradScaler(device=device) if device.type == 'cuda' else None

# Define training parameters for the transfer learning phase.
epoch_transfer_learning = 100  # Train for up to 100 epochs during transfer learning.
epoch_transfer_learning_patience = 10  # Early stopping patience: stop training if no improvement for 10 epochs.
lr_transfer_learning_patience = 5  # If no improvement for 5 epochs, reduce the learning rate.

# Define training parameters for the fine-tuning phase.
epoch_fine_tuning = 100  # Train for up to 100 epochs during fine-tuning.
epoch_fine_tuning_patience = 10  # Early stopping patience for fine-tuning.
lr_fine_tuning_patience = 5  # Learning rate patience for fine-tuning.

In [ ]:
#@title Transfer Learning
# Transfer Learning: Freeze backbone
# The backbone (ResNet-101 in DeepLabV3) is frozen here to prevent its weights from being updated during training.
# This is a common practice in transfer learning where the base (pre-trained) layers are kept fixed,
# & only the newly added layers (like the segmentation head) are trained.
# Setting 'requires_grad = False' ensures the gradients for the backbone are not computed, saving memory & computation.
for param in model.backbone.parameters():
    param.requires_grad = False  # Freeze backbone layers to prevent updates during transfer learning.

# Print statement indicating that the transfer learning phase has started.
print("\nTransfer learning is started!\n")

# Call the 'training' function to perform the transfer learning phase.
# The model, along with key components like the optimizer, loss function, scheduler, & DataLoader, are passed to the function.
# This function will train only the classifier head of the DeepLabV3 model while keeping the backbone frozen.
# Key arguments passed to the training function:
# - 'model': The model being trained.
# - 'train_loader': The DataLoader for the training set.
# - 'epoch_transfer_learning': Number of epochs to run the transfer learning phase.
# - 'optimizer_transfer_learning': The optimizer used for updating model weights during transfer learning.
# - 'criterion': The loss function (CrossEntropyLoss) used for the segmentation task.
# - 'scheduler_transfer_learning': Learning rate scheduler that reduces the learning rate when the validation loss plateaus.
# - 'device': Specifies whether to train on GPU or CPU.
# - 'best_loss': Initialized to infinity for tracking the best validation loss during training.
# - 'lr_min': The minimum learning rate; the scheduler won't reduce the learning rate below this value.
# - 'validation_loader': DataLoader for the validation set, used to evaluate model performance during training.
# - 'epoch_patience': Number of epochs to wait before early stopping if validation loss doesn't improve.
# - 'lr_patience': Number of epochs to wait before reducing the learning rate if no improvement in validation loss.
# - 'model_name': Name of the model for saving checkpoints during training.
# - 'scaler': Used for mixed precision training to reduce memory usage & improve performance when training on GPU.
model, history_transfer_learning = training(model,
                                            train_loader,
                                            epoch_transfer_learning,
                                            optimizer_transfer_learning,
                                            criterion,
                                            scheduler=scheduler_transfer_learning,
                                            device=device,
                                            best_loss=float('inf'),
                                            lr_min=1e-5,
                                            validation_loader=validation_loader,
                                            epoch_patience=epoch_transfer_learning_patience,
                                            lr_patience=lr_transfer_learning_patience,
                                            model_name='model_transfer_learning',
                                            scaler=scaler)

# Print statement indicating that the transfer learning phase has completed.
print("\nTransfer learning is completed!\n")

# Links:
# - https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
# - https://pytorch.org/vision/stable/models.html#torchvision.models.segmentation.deeplabv3_resnet101
# - https://pytorch.org/docs/stable/optim.html#torch.optim.lr_scheduler.ReduceLROnPlateau
# - https://pytorch.org/docs/stable/amp.html

In [ ]:
#@title Fine-Tuning

# Unfreeze Backbone
# In the fine-tuning phase, we enable gradient computation for the backbone of the model (ResNet-101),
# allowing its weights to be updated during training.
# This is necessary for optimizing the entire model (both backbone & classifier head).
for param in model.backbone.parameters():
    param.requires_grad = True  # Unfreeze backbone layers.

print("\nFine-tuning is started!\n")

# Begin fine-tuning the model with the training function.
# Arguments:
# - model: The full model (DeepLabV3 with an unfrozen ResNet-101 backbone).
# - train_loader: DataLoader for training data.
# - epoch_fine_tuning: Number of epochs to train the model during fine-tuning.
# - optimizer_fine_tuning: Optimizer with a lower learning rate, typically used in fine-tuning.
# - criterion: Loss function, CrossEntropyLoss with class weights to handle imbalanced classes.
# - scheduler_fine_tuning: ReduceLROnPlateau scheduler to adjust the learning rate based on validation loss.
# - device: Either "cuda" (GPU) or "cpu" depending on availability.
# - best_loss: Keeps track of the best validation loss during fine-tuning.
# - lr_min: Minimum learning rate for the learning rate scheduler.
# - validation_loader: DataLoader for the validation data, used to monitor performance.
# - epoch_patience: Early stopping patience, stops training if validation loss doesn't improve.
# - lr_patience: Patience for reducing the learning rate if validation loss plateaus.
# - model_name: Filename for saving the model checkpoint during fine-tuning.
# - scaler: Gradient scaler for mixed precision training on GPUs.
model, history_fine_tuning = training(model,
                                      train_loader,
                                      epoch_fine_tuning,
                                      optimizer_fine_tuning,
                                      criterion,
                                      scheduler=scheduler_fine_tuning,
                                      device=device,
                                      best_loss=float('inf'),
                                      lr_min=1e-5,
                                      validation_loader=validation_loader,
                                      epoch_patience=epoch_fine_tuning_patience,
                                      lr_patience=lr_fine_tuning_patience,
                                      model_name='model_fine_tuning',
                                      scaler=scaler)

# Print completion message.
print("\nFine-tuning is completed!\n")

# Links:
# - https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
# - https://pytorch.org/vision/stable/models.html#torchvision.models.segmentation.deeplabv3_resnet101
# - https://pytorch.org/docs/stable/optim.html#torch.optim.lr_scheduler.ReduceLROnPlateau
# - https://pytorch.org/docs/stable/amp.html

In [ ]:
#@title Evaluation
# Evaluation & Visualization

# Function to visualize the original image, ground truth mask, & predicted mask side by side.
def show_image(image, true_mask, pred_mask, palette):
    # VOC_LABELS contains the class names from the PASCAL VOC dataset. This is used to create the legend.
    VOC_LABELS = [
        'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
        'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog',
        'horse', 'motorbike', 'person', 'pottedplant', 'sheep',
        'sofa', 'train', 'tvmonitor'
    ]

    # Create a figure with three subplots to display the original image, ground truth mask, & predicted mask.
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(9, 3))

    # Display the original image
    ax1.imshow(image)
    ax1.axis('off')  # Remove axis ticks
    ax1.set_title('Original Image', fontsize=10)

    # Display the ground truth mask
    ax2.imshow(true_mask, cmap="tab20", interpolation='nearest')  # cmap="tab20" provides a color map for segmentation masks
    ax2.axis('off')  # Remove axis ticks
    ax2.set_title('Ground Truth Mask', fontsize=10)

    # Display the predicted mask
    ax3.imshow(pred_mask, cmap="tab20", interpolation='nearest')
    ax3.axis('off')  # Remove axis ticks
    ax3.set_title('Predicted Mask', fontsize=10)

    # Create a legend for the classes based on the VOC_LABELS & corresponding palette colors
    legend_elements = [
        mpatches.Patch(facecolor=np.array(palette[i]) / 255, label=voc_label) for i, voc_label in enumerate(VOC_LABELS)
    ]

    # Add the legend below the image
    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.50, -0.1), ncol=7, fontsize=8)

    # Display the final figure
    plt.show()

# Take a batch of validation images to visualize predictions
sample = next(iter(validation_loader))
images, masks = sample[0].to(device), sample[1].to(device)  # Move images & masks to the appropriate device (GPU/CPU)

# Generate a random color palette for the segmentation classes.
palette = np.random.randint(0, 255, (model.classifier[-1].out_channels, 3), dtype=np.uint8)
palette[0] = np.array([128, 128, 128])  # Set a default color for the background class

# Disable gradient calculation to speed up evaluation (inference mode)
with torch.no_grad():
    outputs = model(images)['out']  # Run the model & get the output predictions for the images

# Loop over the images & visualize up to 10 predictions
for i in range(min(images.size(0), 10)):
    # Convert the predicted output to a mask by taking the class with the highest score for each pixel
    output_mask = torch.argmax(outputs[i], dim=0).cpu().numpy()  # Get the predicted mask
    output_mask[output_mask == 255] = 0  # Ignore the 255 (void) class in VOC & treat it as background
    output_mask = palette[output_mask]  # Apply the color palette to the predicted mask

    # Convert the true mask to a NumPy array & apply the color palette
    true_mask = masks[i].cpu().numpy()
    true_mask[true_mask == 255] = 0  # Handle the "ignore" class as background
    true_mask = palette[true_mask]  # Apply the palette to the true mask

    # Convert the image tensor to NumPy format & unnormalize it (reverse the normalization applied during preprocessing)
    image_np = images[i].permute(1, 2, 0).cpu().numpy()  # Rearrange dimensions to (H, W, C) for display
    image_np = (image_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))  # Unnormalize the image
    image_np = np.clip(image_np, 0, 1)  # Ensure the pixel values are between [0, 1] for display

    # Call the show_image function to display the original image, ground truth mask, & predicted mask
    show_image(image_np, true_mask, output_mask, palette)

# Relevant links:
# - PASCAL VOC Dataset: http://host.robots.ox.ac.uk/pascal/VOC/
# - DeepLabV3 Model: https://pytorch.org/vision/stable/models.html#torchvision.models.segmentation.deeplabv3_resnet101
# - PyTorch torch.no_grad(): https://pytorch.org/docs/stable/generated/torch.no_grad.html
# - Matplotlib colormaps: https://matplotlib.org/stable/tutorials/colors/colormaps.html

## **2.Oxford-IIIT Pet Experiment**

In [ ]:
#@title Import Necessary Libraries
# Importing necessary libraries for object detection, segmentation, & model training

# Torch (https://pytorch.org/) is a popular machine learning library for deep learning applications
import torch
import torch.optim as optim  # Optimizers such as Adam, SGD for training
import torch.nn as nn  # Neural network components like layers, loss functions

# Torchvision (https://pytorch.org/vision/stable/index.html) is a library containing datasets, models, & transforms
import torchvision
import torchvision.transforms as T  # For data transformations like normalization, resizing, augmentations

# Matplotlib (https://matplotlib.org/) is a plotting library used for visualization
import matplotlib.pyplot as plt  # Used for plotting graphs, images, etc.
import matplotlib.patches as mpatches  # Used for adding colored legends or patches to plots

# NumPy (https://numpy.org/) is a fundamental package for array processing in Python
import numpy as np  # Useful for matrix operations & handling numerical data

# tqdm (https://tqdm.github.io/) provides a progress bar for loops, making training visualization easier
from tqdm.notebook import tqdm  # Specifically designed for Jupyter/Colab notebooks to show progress bars during loops

# DeepLabHead is the final segmentation head of DeepLabV3 (https://arxiv.org/abs/1706.05587), a popular semantic segmentation model
from torchvision.models.segmentation.deeplabv3 import DeepLabHead  # Imports the custom head for DeepLabV3 segmentation models

# Sklearn (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.jaccard_score.html) is a machine learning library;
# here, we use the Jaccard Index (Intersection over Union - IoU) for segmentation performance metrics
from sklearn.metrics import jaccard_score

# defaultdict (https://docs.python.org/3/library/collections.html#collections.defaultdict) is used for handling dictionaries with default values
from collections import defaultdict  # Useful for managing missing keys in a dictionary by providing default values

#Typing module for adding type hints to improve code readability & accuracy
from typing import *  # Provides aliases like List, Tuple, Dict, etc., for type hints


In [ ]:
#@title Define Transformation Functions for the Segmentation Experiment
# Define transformation functions with data augmentation & proper mask handling

def transform_mask(mask: torch.Tensor) -> torch.Tensor:
    """
    Transforms the segmentation mask for multi-class classification.

    This function resizes the input mask to (256, 256) using nearest neighbor
    interpolation (preserving discrete labels) & maps class labels:
        - Class 1 (pet) to 0
        - Class 2 (background) to 1
        - Class 3 (outline) to 2

    Args:
        mask (torch.Tensor): The input segmentation mask.

    Returns:
        torch.Tensor: Transformed mask with mapped class labels & resized to (256, 256).
    """
    # Resize the mask to (256, 256), important to use NEAREST interpolation for masks
    mask = T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST)(mask)

    # Convert the resized mask to a numpy array for easier manipulation of values
    mask = np.array(mask)

    # Adjust class labels: shift class IDs from 1, 2, 3 to 0, 1, 2 for PyTorch compatibility
    mask = mask - 1

    # Return the transformed mask as a PyTorch tensor, dtype=torch.long for discrete labels
    return torch.tensor(mask, dtype=torch.long)

def transform_mask_no_aug(mask: torch.Tensor) -> torch.Tensor:
    """
    Transforms the segmentation mask without augmentation, used for operations like
    class weight calculation. Maps class labels:
        - Class 1 (pet) to 0
        - Class 2 (background) to 1
        - Class 3 (outline) to 2

    Args:
        mask (torch.Tensor): The input segmentation mask.

    Returns:
        torch.Tensor: Transformed mask with mapped class labels & resized to (256, 256).
    """
    # Similar to transform_mask, but without any augmentation (used for more stable calculations)
    mask = T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST)(mask)

    # Convert to numpy array for label manipulation
    mask = np.array(mask)

    # Map class labels: 1 -> 0, 2 -> 1, 3 -> 2
    mask = mask - 1

    # Return transformed mask as torch tensor, dtype=torch.long
    return torch.tensor(mask, dtype=torch.long)


# Image transformation pipeline with augmentations (color jittering) for training data
transform_image = T.Compose([
    # Resize image to 256x256 to match mask dimensions
    T.Resize((256, 256)),

    # Apply random changes to brightness, contrast, saturation, & hue for data augmentation
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),

    # Convert the image to a tensor for further processing
    T.ToTensor(),

    # Normalize the image based on mean & std used in pre-trained models like ResNet
    # Mean & std values are typically ImageNet statistics
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
#@title Essential Functions & Classes
# Initialize model weights using Kaiming Normal initialization
def initialize_weights(m: nn.Module) -> None:
    """
    Initializes the weights of a given neural network layer. Specifically,
    it applies Kaiming Normal initialization for Conv2D layers, which is suitable
    for layers with ReLU activations, & sets biases to zero if they exist.

    Args:
        m (nn.Module): A layer from a neural network.
    """
    # Check if the layer is a Conv2d layer
    if isinstance(m, nn.Conv2d):
        # Apply Kaiming Normal initialization to the weights
        nn.init.kaiming_normal_(m.weight)
        # Set bias to zero if bias exists
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

# Calculate class weights based on the frequency of each class in the dataset
def calculate_class_weights(dataset, num_classes: int = 3) -> torch.Tensor:
    """
    Calculates the class weights for imbalanced datasets by analyzing the number
    of pixels for each class in the dataset. These weights are useful to balance
    the loss function when there are class imbalances.

    Args:
        dataset (torch.utils.data.Dataset): The dataset from which to calculate the class weights.
        num_classes (int): The number of classes in the segmentation task (default: 3).

    Returns:
        torch.Tensor: Class weights as a tensor, to be applied during loss computation.
    """
    # Dictionary to count the number of pixels for each class
    class_counts = defaultdict(int)

    # Loop through each sample in the dataset
    for _, mask in tqdm(dataset, desc="Calculating class weights"):
        mask = mask.numpy()  # Convert the mask to a NumPy array for pixel-wise operations
        # Count pixels for each class
        for cls in range(num_classes):
            class_counts[cls] += np.sum(mask == cls)  # Increment count of class 'cls'

    # Total number of pixels in the dataset
    total_pixels = sum(class_counts.values())

    # List to store the calculated class weights
    class_weights = []
    for cls in range(num_classes):
        # If there are pixels for the class, calculate weight, otherwise default to 1
        if class_counts[cls] > 0:
            class_weights.append(total_pixels / (num_classes * class_counts[cls]))
        else:
            class_weights.append(1.0)  # Handle division by zero for rare classes

    # Convert the class weights to a PyTorch tensor & move to the correct device
    return torch.tensor(class_weights, dtype=torch.float32).to(device)

# Calculate the Mean Intersection over Union (mIoU) between predictions & targets
def calculate_mIoU(preds: torch.Tensor, targets: torch.Tensor, num_classes: int = 3) -> float:
    """
    Calculates the Mean Intersection over Union (mIoU) between predicted masks & target masks.
    mIoU is a common metric for evaluating segmentation models.

    Args:
        preds (torch.Tensor): The predicted segmentation mask.
        targets (torch.Tensor): The ground truth segmentation mask.
        num_classes (int): The number of classes (default: 3).

    Returns:
        float: The mIoU score, averaged over all classes.
    """
    # Flatten both predictions & targets to perform element-wise comparison
    preds = preds.cpu().numpy().flatten()
    targets = targets.cpu().numpy().flatten()

    # Compute the Jaccard Index (IoU) for each class & average (macro average)
    return jaccard_score(targets, preds, average='macro', zero_division=0, labels=list(range(num_classes)))

# Evaluate the model on the validation dataset
def evaluate_model(validation_loader: torch.utils.data.DataLoader, model: nn.Module) -> tuple[float, float]:
    """
    Evaluates the model's performance on the validation dataset. Computes both the
    average loss & the mIoU (Mean Intersection over Union) across all validation samples.

    Args:
        validation_loader (torch.utils.data.DataLoader): DataLoader for the validation set.
        model (nn.Module): The trained segmentation model to evaluate.

    Returns:
        tuple: A tuple containing the average validation loss & the average mIoU.
    """
    # Set the model to evaluation mode (disables dropout, batchnorm, etc.)
    model.eval()

    # Initialize variables for accumulating loss & mIoU
    validation_loss = 0.0
    total_mIoU = 0.0

    # Disable gradient calculation during evaluation (saves memory & computations)
    with torch.no_grad():
        # Iterate over the validation dataset
        for images, masks in validation_loader:
            # Move images & masks to the device (e.g., GPU) & ensure correct data types
            images, masks = images.to(device).float(), masks.to(device).long()

            # Perform a forward pass through the model
            outputs = model(images)['out']

            # Calculate the loss between the predicted outputs & true masks
            loss = criterion(outputs, masks)
            validation_loss += loss.item()  # Accumulate loss

            # Get the class with the highest score (predictions) from the model output
            preds = torch.argmax(outputs, dim=1)

            # Calculate mIoU for the current batch & accumulate
            total_mIoU += calculate_mIoU(preds, masks)

    # Compute the average loss & mIoU over the entire validation set
    avg_loss = validation_loss / len(validation_loader)
    avg_mIoU = total_mIoU / len(validation_loader)

    return avg_loss, avg_mIoU

# Custom Dataset for Oxford-IIIT Pet Dataset, with transformations applied to both images & masks
class OxfordIIITPetDataset(torchvision.datasets.OxfordIIITPet):
    """
    Custom Dataset for the Oxford-IIIT Pet dataset, overriding the __getitem__ method
    to apply separate transformations to both the input images & the segmentation masks.

    Args:
        *args: Positional arguments for the parent class constructor.
        **kwargs: Keyword arguments for the parent class constructor, including
                  'transform' for image transformations & 'target_transform' for mask transformations.
    """
    def __init__(self, *args, **kwargs):
        # Pop transformation functions for the image & mask from kwargs
        self.transform_image = kwargs.pop('transform', None)
        self.transform_mask = kwargs.pop('target_transform', None)

        # Initialize the parent class (torchvision.datasets.OxfordIIITPet) with segmentation targets
        super(OxfordIIITPetDataset, self).__init__(*args, target_types="segmentation", **kwargs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Retrieves the image & mask at the specified index, applies the defined transformations
        to both, & returns the processed image & mask.

        Args:
            index (int): Index of the item to retrieve.

        Returns:
            tuple: Transformed image & corresponding transformed mask.
        """
        # Fetch the original image & segmentation mask using the parent class method
        image, mask = super(OxfordIIITPetDataset, self).__getitem__(index)

        # Apply the image transformation if provided
        if self.transform_image:
            image = self.transform_image(image)

        # Apply the mask transformation if provided
        if self.transform_mask:
            mask = self.transform_mask(mask)

        # Return the transformed image & mask as a tuple
        return image, mask


In [ ]:
#@title Training Function
def training(model: nn.Module,
             train_loader: torch.utils.data.DataLoader,
             num_epochs: int,
             optimizer: optim.Optimizer,
             criterion: nn.Module,
             scheduler: Optional[optim.lr_scheduler._LRScheduler] = None,
             device: str = "cuda",
             best_loss: float = float('inf'),
             validation_loader: Optional[torch.utils.data.DataLoader] = None,
             epoch_patience: Optional[int] = None,
             lr_patience: Optional[int] = None,
             model_name: Optional[str] = None,
             scaler: Optional[torch.cuda.amp.GradScaler] = None) -> Tuple[nn.Module, Dict[str, List[float]]]:
    """
    Trains a segmentation model, optionally with mixed precision, & applies early stopping
    & learning rate scheduling. Tracks training/validation loss & validation mIoU.

    Args:
        model (nn.Module): The PyTorch model to be trained.
        train_loader (torch.utils.data.DataLoader): DataLoader for the training set.
        num_epochs (int): Number of epochs to train the model.
        optimizer (optim.Optimizer): Optimizer for updating the model weights.
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss).
        scheduler (Optional[optim.lr_scheduler._LRScheduler]): Learning rate scheduler.
        device (str): Device to train the model on ('cuda' or 'cpu').
        best_loss (float): Best validation loss for early stopping.
        validation_loader (Optional[torch.utils.data.DataLoader]): DataLoader for the validation set.
        epoch_patience (Optional[int]): Number of epochs to wait before early stopping if no improvement.
        lr_patience (Optional[int]): Patience for learning rate reduction.
        model_name (Optional[str]): Name to save the model.
        scaler (Optional[torch.cuda.amp.GradScaler]): Scaler for mixed precision training.

    Returns:
        Tuple[nn.Module, Dict[str, List[float]]]: The trained model & training history, including losses & mIoU.
    """

    if epoch_patience:
        epoch_patience_counter = 0  # Tracks how many epochs we've gone without improvement

    if lr_patience:
        lr_patience_counter = 0  # Tracks learning rate patience for LR scheduling

    train_loss = []  # Keeps track of the average training loss for each epoch
    validation_loss = []  # Keeps track of the validation loss for each epoch (if validation is used)
    validation_mIoU = []  # Keeps track of the mIoU for validation set (if validation is used)

    # Loop through each epoch
    for epoch in range(num_epochs):
        running_loss = 0.0
        model.train()  # Set the model to training mode

        # Set up progress bar for the training loop
        progress_bar = tqdm(train_loader,
                            desc=f"Epoch {epoch + 1}/{num_epochs}, LR: {optimizer.param_groups[0]['lr']:.6f}",
                            unit="batch",
                            leave=True,
                            dynamic_ncols=True)

        # Training loop
        for i, (images, masks) in enumerate(progress_bar, start=1):
            images = images.to(device).float()  # Move images to the device (GPU/CPU)
            masks = masks.to(device).long()  # Move masks to the device & cast to long for classification

            optimizer.zero_grad()  # Zero the gradients before each iteration

            if scaler:  # If using mixed precision training
                with torch.amp.autocast(device_type=device.type):
                    outputs = model(images)['out']  # Forward pass
                    loss = criterion(outputs, masks)  # Compute loss
                scaler.scale(loss).backward()  # Scale the loss for mixed precision
                scaler.unscale_(optimizer)  # Unscale gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
                scaler.step(optimizer)  # Perform optimizer step
                scaler.update()  # Update the scaler for the next iteration
            else:  # Standard training (no mixed precision)
                outputs = model(images)['out']  # Forward pass
                loss = criterion(outputs, masks)  # Compute loss
                loss.backward()  # Backward pass
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
                optimizer.step()  # Perform optimizer step

            running_loss += loss.item()  # Accumulate the running loss
            ave_loss = running_loss / i  # Compute the average loss for the current epoch
            progress_bar.set_postfix(loss=f"{ave_loss:.5f}")  # Update the progress bar with the average loss

        train_loss.append(ave_loss)  # Append the average loss for this epoch

        # Validation loop (if a validation_loader is provided)
        if validation_loader:
            val_loss, val_mIoU = evaluate_model(validation_loader, model)  # Evaluate the model on the validation set
            validation_loss.append(val_loss)  # Append the validation loss
            validation_mIoU.append(val_mIoU)  # Append the validation mIoU

            # Update the progress bar with validation metrics
            progress_bar.set_postfix(loss=f"| Ave: {ave_loss:.5f} | Val: {val_loss:.5f} | mIoU: {val_mIoU:.4f}")

            # Learning rate scheduler
            if scheduler:
                if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(val_loss)  # Step based on validation loss for ReduceLROnPlateau
                else:
                    scheduler.step()  # Standard step for other types of schedulers

            # Early stopping mechanism based on validation loss
            if epoch_patience:
                if val_loss < best_loss:
                    best_loss = val_loss  # Update the best loss
                    epoch_patience_counter = 0  # Reset patience counter
                    # Save the best model
                    name = 'model.pth' if not model_name else f"{model_name}.pth"
                    torch.save(model.state_dict(), name)
                else:
                    epoch_patience_counter += 1  # Increment patience counter

                if epoch_patience_counter >= epoch_patience:  # Check for early stopping condition
                    print(f"Early stopping at epoch {epoch + 1}")
                    break  # Stop training if no improvement

    # Save the final model at the end of training
    name = 'model_final.pth' if not model_name else f"{model_name}_final.pth"
    torch.save(model.state_dict(), name)

    # Return the final model & training history
    history = {
        'training_loss': train_loss,
        'validation_loss': validation_loss,
        'validation_mIoU': validation_mIoU
    }

    return model, history


In [ ]:
#@title Dataset Preparation

# Set batch sizes for training & validation
training_batch_size = 8  # Number of samples per batch for training
validation_batch_size = 8  # Number of samples per batch for validation

# Load the Oxford-IIIT Pet Dataset with augmentations applied to both images & masks
dataset_train_aug = OxfordIIITPetDataset(
    root="./",  # Root directory to store the dataset
    split="trainval",  # Use the combined train/validation split
    download=True,  # Download the dataset if not already available
    transform=transform_image,  # Apply image transformations such as resizing & color jitter
    target_transform=transform_mask  # Apply mask transformations for segmentation
)

# Load the dataset again but without augmentations to calculate class weights
dataset_class_weights = OxfordIIITPetDataset(
    root="./",  # Root directory to store the dataset
    split="trainval",  # Use the combined train/validation split
    download=False,  # Dataset is already downloaded
    transform=None,  # No image transformations needed for class weight calculation
    target_transform=transform_mask_no_aug  # Apply mask transformations without augmentations
)

# Split the dataset into 80% for training & 20% for validation
train_size = int(0.8 * len(dataset_train_aug))  # 80% of data for training
validation_size = len(dataset_train_aug) - train_size  # Remaining 20% for validation
train_dataset, validation_dataset = torch.utils.data.random_split(
    dataset_train_aug, [train_size, validation_size]
)  # Randomly split the dataset into training & validation sets

# Create DataLoader for the training dataset
train_loader = torch.utils.data.DataLoader(
    train_dataset,  # The training dataset after the split
    batch_size=training_batch_size,  # Number of samples per batch
    shuffle=True,  # Shuffle data at every epoch to reduce overfitting
    num_workers=4,  # Number of subprocesses to use for data loading
    pin_memory=True  # Pin memory for faster data transfer to GPU
)

# Create DataLoader for the validation dataset
validation_loader = torch.utils.data.DataLoader(
    validation_dataset,  # The validation dataset after the split
    batch_size=validation_batch_size,  # Number of samples per batch
    shuffle=False,  # No need to shuffle validation data
    num_workers=4,  # Number of subprocesses for data loading
    pin_memory=True  # Pin memory for faster data transfer to GPU
)

In [ ]:
#@title Model Development & Preparation
# Model setup with DeepLabV3 ResNet101 for improved accuracy
# DeepLabV3 (https://arxiv.org/abs/1706.05587) is a state-of-the-art model for semantic segmentation.
# Here, ResNet101 (https://arxiv.org/abs/1512.03385) is used as the backbone for feature extraction.

# Set the device to use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Number of classes for the segmentation task: 0 (pet), 1 (background), 2 (outline)
num_classes = 3  # Multi-class segmentation

# Import the DeepLabV3 model with ResNet101 backbone & default pre-trained weights (ImageNet-trained)
from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights

# Initialize the DeepLabV3 model with ResNet101 backbone & pre-trained weights
model = deeplabv3_resnet101(weights=DeepLabV3_ResNet101_Weights.DEFAULT)

# Modify the classifier head of the model to output 'num_classes' predictions (3 classes in our case)
# The '2048' refers to the number of input features from the ResNet101 backbone
model.classifier = DeepLabHead(2048, num_classes)

# Initialize the weights of the classifier head using Kaiming Normal initialization
model.classifier.apply(initialize_weights)

# Move the model to the specified device (GPU/CPU)
model.to(device)

# Calculate class weights to handle class imbalance
# Class weights adjust the loss function to give higher importance to underrepresented classes
class_weights = calculate_class_weights(dataset_class_weights, num_classes=num_classes)

# Use CrossEntropyLoss for multi-class classification. The class weights are passed to the loss function.
# This ensures that the loss is calculated considering class imbalances in the dataset.
criterion = nn.CrossEntropyLoss(weight=class_weights)

# --------------------------------------------
# Optimizers & Learning Rate Schedulers
# --------------------------------------------

# Transfer learning setup:
# Optimizer: Stochastic Gradient Descent (SGD) with momentum for faster convergence & weight decay for regularization
# Only parameters that require gradients are optimized (useful if some layers are frozen during transfer learning)
optimizer_transfer_learning = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),  # Filter parameters that require gradients
    lr=1e-3,  # Learning rate for transfer learning phase
    momentum=0.9,  # Momentum helps accelerate gradients vectors in the right direction
    weight_decay=1e-4  # Weight decay (L2 regularization) to prevent overfitting
)

# Learning rate scheduler for transfer learning phase:
# ReduceLROnPlateau decreases the learning rate when validation loss plateaus.
scheduler_transfer_learning = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_transfer_learning,
    mode='min',  # Reduce LR when a monitored quantity (validation loss) has stopped decreasing
    factor=0.5,  # Multiply LR by this factor when the condition is met
    patience=5  # Wait for 5 epochs before reducing the LR
)

# Fine-tuning setup:
# After the initial transfer learning, we fine-tune the entire model with a smaller learning rate.
optimizer_fine_tuning = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),  # Parameters that require gradients
    lr=1e-4,  # Lower learning rate for fine-tuning phase
    momentum=0.9,  # Use momentum for faster & stable convergence
    weight_decay=1e-4  # Apply weight decay for regularization
)

# Learning rate scheduler for fine-tuning phase:
scheduler_fine_tuning = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fine_tuning,
    mode='min',  # Reduce LR when the validation loss stops improving
    factor=0.5,  # Halve the learning rate if validation loss plateaus
    patience=3  # Wait for 3 epochs before reducing the learning rate
)

# --------------------------------------------
# Patience & Epoch Parameters
# --------------------------------------------

# Transfer learning phase:
# Maximum epochs for transfer learning & patience for early stopping & LR scheduling
epoch_transfer_learning = 100  # Number of epochs for transfer learning phase
epoch_transfer_learning_patience = 10  # Early stopping if no improvement after 10 epochs
lr_transfer_learning_patience = 5  # LR is reduced if no improvement after 5 epochs

# Fine-tuning phase:
# Maximum epochs for fine-tuning & patience for early stopping & LR scheduling
epoch_fine_tuning = 100  # Number of epochs for fine-tuning phase
epoch_fine_tuning_patience = 10  # Early stopping if no improvement after 10 epochs
lr_fine_tuning_patience = 5  # LR is reduced if no improvement after 5 epochs

In [ ]:
#@title Transfer Learning

# Initialize GradScaler for mixed precision training (optional)
# Mixed precision training helps to reduce memory usage & improve computational efficiency.
# The GradScaler is used to scale the loss to prevent underflow during backpropagation when using float16 precision.
# This is only necessary when using CUDA-enabled GPUs.
scaler = torch.amp.GradScaler() if device.type == 'cuda' else None

# Transfer Learning: Freeze the backbone layers of the model
# The backbone consists of the feature extraction layers, typically pretrained on a large dataset like ImageNet.
# Freezing the backbone means we only update the classifier head during the initial phase of transfer learning.
for param in model.backbone.parameters():
    param.requires_grad = False  # Disable gradients for all backbone parameters

print("\nTransfer learning is started!\n")

# Call the training function for the transfer learning phase
# Arguments:
# - model: The DeepLabV3 segmentation model with ResNet101 backbone.
# - train_loader: DataLoader for the training dataset.
# - epoch_transfer_learning: Number of epochs for the transfer learning phase.
# - optimizer_transfer_learning: Optimizer (SGD with momentum) used for training the model.
# - criterion: Loss function (CrossEntropyLoss with class weights).
# - scheduler_transfer_learning: Learning rate scheduler to adjust LR based on validation performance.
# - device: The device (GPU or CPU) to perform training on.
# - best_loss: Initial best loss for tracking improvements (used for saving the best model).
# - validation_loader: DataLoader for the validation dataset.
# - epoch_patience: Number of epochs to wait for improvement before early stopping.
# - lr_patience: Number of epochs to wait for learning rate reduction.
# - model_name: Name under which to save the best model weights.
# - scaler: GradScaler for mixed precision training.

model, history_transfer_learning = training(
    model=model,
    train_loader=train_loader,
    num_epochs=epoch_transfer_learning,
    optimizer=optimizer_transfer_learning,
    criterion=criterion,
    scheduler=scheduler_transfer_learning,
    device=device,
    best_loss=float('inf'),  # Starting with an infinite loss to ensure improvement tracking
    validation_loader=validation_loader,
    epoch_patience=epoch_transfer_learning_patience,  # Patience for early stopping
    lr_patience=lr_transfer_learning_patience,  # Patience for learning rate reduction
    model_name='model_transfer_learning',  # File name for saving the model weights
    scaler=scaler  # Mixed precision training scaler (if using CUDA)
)

print("\nTransfer learning is completed!\n")

In [ ]:
#@title Fine-Tuning

# Fine-Tuning: Unfreeze the backbone
# In the fine-tuning phase, we unfreeze the backbone layers so that they can be updated during training.
# This allows the entire model, including the feature extractor (ResNet101 backbone), to adapt to the specific task (segmentation).
for param in model.backbone.parameters():
    param.requires_grad = True  # Enable gradients for all backbone parameters

print("\nFine-tuning is started!\n")

# Call the training function for the fine-tuning phase
# - During fine-tuning, the entire model (backbone + classifier head) is trained.
# - A lower learning rate is typically used to prevent large changes to the pre-trained weights.
model, history_fine_tuning = training(
    model=model,  # The full DeepLabV3 model with the unfrozen ResNet101 backbone
    train_loader=train_loader,  # DataLoader for the training dataset
    num_epochs=epoch_fine_tuning,  # Number of epochs for fine-tuning phase
    optimizer=optimizer_fine_tuning,  # Optimizer for fine-tuning (lower learning rate)
    criterion=criterion,  # Loss function (CrossEntropyLoss with class weights)
    scheduler=scheduler_fine_tuning,  # Learning rate scheduler for fine-tuning
    device=device,  # The device to train on (GPU or CPU)
    best_loss=float('inf'),  # Track the best validation loss to save the best model
    validation_loader=validation_loader,  # DataLoader for validation set
    epoch_patience=epoch_fine_tuning_patience,  # Early stopping patience for fine-tuning
    lr_patience=lr_fine_tuning_patience,  # Learning rate reduction patience
    model_name='model_fine_tuning',  # Name to save the fine-tuned model
    scaler=scaler  # Mixed precision training scaler (optional)
)

print("\nFine-tuning is completed!\n")


In [ ]:
#@title Evaluation & Visualization

# Function to display the original image, ground truth mask, & predicted mask side by side
def show_image(image: np.ndarray, true_mask: np.ndarray, pred_mask: np.ndarray, palette: np.ndarray) -> None:
    """
    Displays the original image, ground truth mask, & predicted mask in a row of subplots.

    Args:
        image (np.ndarray): The original image in RGB format.
        true_mask (np.ndarray): Ground truth segmentation mask.
        pred_mask (np.ndarray): Predicted segmentation mask from the model.
        palette (np.ndarray): Color palette for the segmentation classes.
    """

    # Define the labels for the segmentation classes
    PET_LABELS = ['pet', 'background', 'outline']

    # Create a figure with 3 subplots: original image, ground truth mask, predicted mask
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(9, 3))

    # Display the original image
    ax1.imshow(image)
    ax1.axis('off')  # Remove axis for a cleaner look
    ax1.set_title('Original Image', fontsize=10)

    # Display the ground truth segmentation mask
    ax2.imshow(true_mask, cmap="tab20", interpolation='nearest')
    ax2.axis('off')  # Remove axis
    ax2.set_title('Ground Truth Mask', fontsize=10)

    # Display the predicted segmentation mask
    ax3.imshow(pred_mask, cmap="tab20", interpolation='nearest')
    ax3.axis('off')  # Remove axis
    ax3.set_title('Predicted Mask', fontsize=10)

    # Create a legend showing the label for each class (pet, background, outline)
    legend_elements = [
        mpatches.Patch(facecolor=np.array(palette[i]) / 255, label=label) for i, label in enumerate(PET_LABELS)
    ]

    # Add the legend below the image with class labels
    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.50, 0), ncol=7, fontsize=8)

    # Display the complete figure with the image & masks
    plt.show()

# Get a batch of validation samples
sample = next(iter(validation_loader))
images, masks = sample[0].to(device), sample[1].to(device)

# Create a random color palette for visualizing the segmentation classes
# 'model.classifier[-1].out_channels' gives the number of output channels (classes) in the model's classifier head
palette = np.random.randint(0, 255, (model.classifier[-1].out_channels, 3), dtype=np.uint8)
palette[0] = np.array([128, 128, 128])  # Assign a specific color (gray) to the first class (e.g., background)

# Forward pass through the model without computing gradients
with torch.no_grad():
    outputs = model(images)['out']  # Get model output (segmentation masks)

# Loop over the first 10 samples in the batch to visualize the results
for i in range(min(images.size(0), 10)):
    # Get the predicted mask by taking the argmax along the class dimension (for each pixel)
    output_mask = torch.argmax(outputs[i], dim=0).cpu().numpy()
    output_mask[output_mask == 255] = 0  # Handle the ignore index (255) in the mask by replacing it with 0
    output_mask = palette[output_mask]  # Map the predicted mask to the color palette

    # Get the ground truth mask & map it to the same color palette
    true_mask = masks[i].cpu().numpy()
    true_mask[true_mask == 255] = 0  # Handle ignore index in ground truth mask
    true_mask = palette[true_mask]

    # Convert the image tensor back to a NumPy array & denormalize it
    image_np = images[i].permute(1, 2, 0).cpu().numpy()  # Convert from (C, H, W) to (H, W, C)
    image_np = (image_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))  # Denormalize
    image_np = np.clip(image_np, 0, 1)  # Clip values to be within valid range [0, 1] for display

    # Display the original image, true mask, & predicted mask using the show_image function
    show_image(image_np, true_mask, output_mask, palette)

## **3. COCO Experiment**

In [ ]:
#@title Import Necessary Libraries

# The `random` module is used to generate random numbers, which can be useful for random sampling or data shuffling.
import random

# `matplotlib.pyplot` is a popular library for creating static, animated, & interactive visualizations.
import matplotlib.pyplot as plt

# `numpy` is a fundamental library for performing numerical operations on arrays & matrices.
import numpy as np

# `torch` is the core library for PyTorch, providing tensor operations & model building tools.
import torch

# The `patches` module from matplotlib allows adding shapes (e.g., rectangles) to visualizations.
from matplotlib import patches

# `to_rgb` is a function from matplotlib that converts color names or hex codes into RGB format.
from matplotlib.colors import to_rgb

# `seaborn` is built on top of matplotlib & offers enhanced visualization capabilities.
import seaborn as sns

# `torch.optim` provides optimization algorithms like SGD & Adam, which are essential for model training.
import torch.optim as optim

# `torchvision` offers utilities like pre-trained models, datasets, & transformations for computer vision tasks.
import torchvision

# Importing a pre-trained Mask R-CNN model with a ResNet50 backbone & Feature Pyramid Network (FPN).
# More details about Mask R-CNN can be found in the paper by He et al. (2017): https://arxiv.org/abs/1703.06870
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

# `MaskRCNNPredictor` allows modification of the prediction head of the Mask R-CNN model, enabling customization for specific tasks (e.g., changing the number of output classes).
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

# `functional` from torchvision provides various image transformation functions like resizing, normalization, & conversion to tensors.
# More about these transformations: https://pytorch.org/vision/stable/transforms.html
from torchvision.transforms import functional as F

# `CocoDetection` is a dataset class in torchvision that loads COCO dataset annotations for object detection tasks.
# The COCO dataset is widely used for object detection, segmentation, & captioning challenges. Official site: https://cocodataset.org/
from torchvision.datasets import CocoDetection

# This module (`T`) includes common image transformations (such as cropping, flipping, & rotating), widely used in preparing data for training.
import torchvision.transforms as T

# `PIL` (Python Imaging Library) provides functionality for opening, manipulating, & saving image files in various formats.
from PIL import Image

# `defaultdict` from the collections module creates a dictionary with a default value when a key is not found.
# This is especially useful for counting or grouping items.
from collections import defaultdict

# `tqdm` is a progress bar library, ideal for tracking loops, especially when working with large datasets or during model training.
# The `notebook` version is tailored for Jupyter notebooks & similar environments like Colab.
from tqdm.notebook import tqdm

# The `os` module provides utilities for interacting with the operating system, such as reading file paths & directories.
import os

# `json` is a lightweight data interchange format, often used to load or save data in COCO annotations or other formats.
import json

# `ListedColormap` is a function from matplotlib that allows creating custom color maps, useful for visualizing segmentation masks with distinct colors.
from matplotlib.colors import ListedColormap

# `COCO` is a Python API provided by `pycocotools`, which is specifically designed for working with the COCO dataset.
# The library assists in loading, parsing, & visualizing COCO annotations.
# If you're using Google Colab or Jupyter, make sure pycocotools is installed with: `!pip install pycocotools`
from pycocotools.coco import COCO

# `typing` is a built-in module for adding type hints to Python code, enhancing code readability & helping avoid type-related errors.
# Example: List[int] indicates a list of integers, Dict[str, int] indicates a dictionary with string keys & integer values.
from typing import *


In [ ]:
#@title Define Transformation Functions

# `ComposeTransforms` is a custom class that applies a sequence of transformations to both an input image
# & its corresponding target (e.g., bounding boxes, segmentation masks).
# It is similar to `torchvision.transforms.Compose` but specifically designed to transform both images & annotations.
class ComposeTransforms:

    '''
    Constructor for the `ComposeTransforms` class.

    Args:
        transforms (List[Callable]): A list of transformation functions that will be applied sequentially
                                     to both the image & the target (annotations).
    '''
    def __init__(self, transforms: List[Callable]):
        self.transforms = transforms

    '''
    The `__call__` method applies all the transformations stored in `self.transforms` to both
    the image & the target.

    Args:
        image (Image.Image): The input image, typically in PIL format, to be transformed.
        target (Dict): The target annotations (bounding boxes, labels, masks, etc.) associated with the image.

    Returns:
        Tuple[torch.Tensor, Dict]: A tuple containing the transformed image (as a tensor) & the transformed target.
    '''
    def __call__(self, image: Image.Image, target: Dict) -> Tuple[torch.Tensor, Dict]:
        # Apply each transformation in the list to the image & target
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

# `ToTensor` is a transformation class that converts images from PIL format to PyTorch tensors.
# Tensors are the expected format for PyTorch models, & this transformation ensures the input
# data is compatible with the model.
class ToTensor:

    '''
    The `__call__` method checks if the image is already a PyTorch tensor. If not, it converts the image from
    a PIL image (or NumPy array) to a tensor using `F.to_tensor()`.

    Args:
        image (Image.Image or np.ndarray): The input image to be converted to a tensor.
        target (Dict): The target annotations (bounding boxes, labels, masks, etc.) associated with the image.

    Returns:
        Tuple[torch.Tensor, Dict]: The image as a PyTorch tensor & the unchanged target.
    '''
    def __call__(self, image: Image.Image, target: Dict) -> Tuple[torch.Tensor, Dict]:
        # If the image is already a tensor, no conversion is needed
        if isinstance(image, torch.Tensor):
            return image, target
        # Convert the image to a tensor & return it along with the unchanged target
        image = F.to_tensor(image)
        return image, target

# `transform` is an instance of `ComposeTransforms` that defines a pipeline of image transformations.
# It only includes the `ToTensor` transformation, which converts input images to tensors.
transform = ComposeTransforms([
    ToTensor()
])


In [ ]:
#@title Define Custom COCO Dataset
'''
This code defines a custom dataset class `COCODataset` that extends from `CocoDetection`,
designed to handle images & annotations in the COCO dataset format, widely used in object detection,
segmentation, & similar computer vision tasks. It processes bounding boxes, segmentation masks,
labels, & other metadata to be compatible with deep learning models.
'''

class COCODataset(CocoDetection):
    '''
    The COCODataset class inherits from the PyTorch torchvision `CocoDetection` class.
    It facilitates the loading & preprocessing of COCO-format datasets for object detection
    & instance segmentation tasks. The class performs the following:

    1. Handles dataset loading with annotations.
    2. Converts annotations (bounding boxes, segmentation masks) to a suitable format.
    3. Applies any image transformations (e.g., resizing, augmentation) if provided.
    4. Maps COCO category IDs to custom label indices.

    Parameters:
    -----------
    root: str
        Path to the root directory where images are stored.
    annFile: str
        Path to the JSON annotation file for the dataset (COCO format).
    transforms: Callable, optional
        Optional data transformations (like resizing, normalization, etc.) applied to the image
        & corresponding target (annotations).
    '''

    def __init__(self, root: str, annFile: str, transforms: Callable = None):
        # Initialize the parent class CocoDetection with root & annotation file
        super(COCODataset, self).__init__(root, annFile)

        # Store the transforms for later use (augmentation, etc.)
        self.transforms = transforms

        # Get all category IDs from the COCO dataset in a sorted order
        self.category_ids = sorted(self.coco.getCatIds())

        # Create a mapping from COCO category IDs to custom label indices (1-based index)
        self.category_id_to_label = {cat_id: idx + 1 for idx, cat_id in enumerate(self.category_ids)}

        # Initialize an empty dictionary to store mappings from labels to category names
        self.label_to_category_name = {}

        # Populate the label_to_category_name dictionary with category names from COCO
        for idx, cat_id in enumerate(self.category_ids):
            cat_info = self.coco.loadCats([cat_id])[0]  # Load category info from COCO API
            self.label_to_category_name[idx + 1] = cat_info['name']  # Map label to category name

        # Build the COCO_INSTANCE_CATEGORY_NAMES list with the background as the first class
        # This list is used to interpret model outputs
        self.COCO_INSTANCE_CATEGORY_NAMES = ['__background__'] + \
                                            [self.label_to_category_name[i] for i in range(1, len(self.category_ids) + 1)]

    def __getitem__(self, idx: int):
        '''
        Retrieve a specific image & its corresponding annotations (target) at a given index.

        This method overrides the __getitem__ method from the parent class `CocoDetection`. It loads the image
        & annotations for a given index, processes bounding boxes, masks, areas, & crowd information,
        & returns the image & target in a format ready for object detection models.

        Parameters:
        -----------
        idx: int
            Index of the image/annotation pair to retrieve.

        Returns:
        --------
        image: PIL.Image
            The image at the specified index.
        target: dict
            A dictionary containing processed annotations, including:
            - "boxes": Bounding boxes in [x1, y1, x2, y2] format.
            - "labels": Class labels corresponding to objects in the image.
            - "masks": Binary masks for each object.
            - "image_id": Image ID.
            - "area": Area of each object in the image.
            - "iscrowd": Whether the object is part of a crowd (boolean).
        '''

        try:
            # Retrieve the image & target annotations from the parent class
            image, targets = super(COCODataset, self).__getitem__(idx)

            # Retrieve the image ID for the current sample
            image_id = self.ids[idx]

            # Get all annotation IDs associated with this image ID
            ann_ids = self.coco.getAnnIds(imgIds=image_id)

            # Load the corresponding annotations (bbox, category_id, segmentation, etc.)
            anns = self.coco.loadAnns(ann_ids)

            # Initialize lists to store information for each object in the image
            boxes = []
            labels = []
            masks = []
            areas = []
            iscrowd = []

            # Loop through each annotation
            for ann in anns:
                # Skip annotations without bounding boxes
                if ann.get('bbox') is None:
                    continue

                # Retrieve the category ID & map it to our custom label index
                category_id = ann['category_id']
                label = self.category_id_to_label[category_id]  # Map category_id to label
                labels.append(label)  # Append label to the list

                # Append the bounding box in COCO format [x, y, width, height]
                boxes.append(ann['bbox'])

                # Convert COCO segmentation annotations to a binary mask & append
                masks.append(self.coco.annToMask(ann))

                # Append the area of the object
                areas.append(ann['area'])

                # Check if the object is part of a crowd
                iscrowd.append(ann.get('iscrowd', 0))  # Default to 0 if not provided

            # If no bounding boxes exist for this sample, skip it
            if not boxes:
                return None

            # Convert bounding boxes from [x, y, width, height] to [x1, y1, x2, y2] format
            boxes = torch.as_tensor(boxes, dtype=torch.float32)  # Convert to tensor
            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]  # x2 = x1 + width
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]  # y2 = y1 + height

            # Ensure the bounding boxes have positive width & height
            if not ((boxes[:, 2] > boxes[:, 0]).all() & (boxes[:, 3] > boxes[:, 1]).all()):
                print(f"Invalid box with negative width or height at index {idx}: {boxes}")
                return None  # Skip samples with invalid bounding boxes

            # Convert all lists to tensors for compatibility with models
            labels = torch.as_tensor(labels, dtype=torch.int64)  # Class labels as int64 tensor
            masks = torch.as_tensor(masks, dtype=torch.uint8)  # Binary masks as uint8 tensor
            areas = torch.as_tensor(areas, dtype=torch.float32)  # Areas as float32 tensor
            iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64)  # iscrowd as int64 tensor

            # Create a dictionary for the target annotations
            target = {}
            target["boxes"] = boxes  # Bounding boxes
            target["labels"] = labels  # Object class labels
            target["masks"] = masks  # Segmentation masks
            target["image_id"] = torch.tensor([image_id])  # Image ID
            target["area"] = areas  # Object areas
            target["iscrowd"] = iscrowd  # Is object part of a crowd?

            # Apply any image & target transformations if they were provided
            if self.transforms is not None:
                image, target = self.transforms(image, target)

            # Return the processed image & target
            return image, target

        except Exception as e:
            # If an error occurs while loading the data, skip the sample & print the error
            print(f"Error loading data at index {idx}: {e}")
            return None  # Skip this sample if there are issues


In [ ]:
#@title Define Collate Function for DataLoader
'''
This function defines a custom `collate_fn` for use in PyTorch's `DataLoader`. The purpose of the `collate_fn` is to handle
how batches of data are combined together when loading them for training or inference. The default behavior of `DataLoader`
can sometimes lead to issues, especially when dealing with object detection or segmentation tasks where the data (e.g., images, annotations)
may have variable shapes or missing data points.

The function:
1. Filters out `None` values from the batch (which might occur if an invalid sample was returned from the dataset).
2. Handles cases where the entire batch may consist of invalid data by returning empty tuples.
3. Groups the valid samples into a tuple of batched data that can be used for model training or inference.

References:
- PyTorch DataLoader Documentation: https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
'''

# Define the `collate_fn` function, which is responsible for combining a batch of samples into a single batch
# for use in training or inference. This function is used to handle variable-sized data or missing samples.
def collate_fn(batch):
    # Filter out `None` values from the batch.
    # This lambda function checks each element in the batch & removes any `None` elements.
    # `None` values can occur if a data sample is deemed invalid (e.g., no annotations or invalid image).
    batch = list(filter(lambda x: x is not None, batch))

    # Check if the entire batch is empty after filtering.
    # If all the samples in the batch were invalid & removed, return empty tuples.
    if len(batch) == 0:
        return (), ()  # Returning empty tuples ensures the program doesn't crash.

    # Use the `zip` function to group corresponding elements of the remaining valid samples.
    # `zip(*batch)` groups the elements by type, for example, grouping all images together & all targets together.
    # `tuple()` converts the zipped object into a tuple of batches (e.g., a tuple of images & a tuple of annotations).
    return tuple(zip(*batch))


In [ ]:
#@title Initialize the Mask R-CNN Model
'''
This function `get_instance_segmentation_model` initializes a Mask R-CNN model, which is used
for both object detection & instance segmentation tasks. Mask R-CNN is a powerful model
that extends Faster R-CNN by adding a branch for predicting segmentation masks for each region of interest (RoI).

The function sets up the model to handle a specific number of classes, replacing the default COCO-trained
classifier & mask heads with new ones customized for the dataset at hand.

References:
- Mask R-CNN paper: https://arxiv.org/abs/1703.06870
- PyTorch documentation for Mask R-CNN: https://pytorch.org/vision/stable/models/mask_rcnn.html
'''

def get_instance_segmentation_model(num_classes: int) -> torchvision.models.detection.MaskRCNN:
    '''
    This function returns a Mask R-CNN model pre-trained on COCO, with modified classifier & mask heads
    to handle a custom number of object categories.

    Parameters:
    -----------
    num_classes: int
        The number of object classes to predict (including the background). For example, for binary
        segmentation (foreground vs background), you would pass `num_classes=2`.

    Returns:
    --------
    model: torchvision.models.detection.MaskRCNN
        A Mask R-CNN model ready to be trained or used for inference on a custom dataset.
    '''

    # Load the Mask R-CNN model pre-trained on the COCO dataset.
    # COCO has 91 classes, but we will modify the model to fit the number of classes in our custom dataset.
    # Documentation for the weights can be found here: https://pytorch.org/vision/stable/models.html#torchvision.models.detection.maskrcnn_resnet50_fpn
    weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT  # Load default COCO-pretrained weights
    model = maskrcnn_resnet50_fpn(weights=weights)   # Initialize the Mask R-CNN model with the weights

    # In Mask R-CNN, the box (classification & bounding box regression) & mask heads are fully connected
    # layers that take features from the backbone network. Here, we replace the box classifier head with a new one
    # that has the appropriate number of output classes.

    # Get the number of input features (hidden units) for the classifier (bounding box head)
    in_features = model.roi_heads.box_predictor.cls_score.in_features  # Number of input features to the final box classifier layer

    # Replace the box predictor with a new one for the number of classes in the custom dataset
    # `FastRCNNPredictor` is a built-in helper class to easily create a new box predictor
    # Documentation: https://pytorch.org/vision/stable/_modules/torchvision/models/detection/faster_rcnn.html#FastRCNNPredictor
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

    # Now, we modify the mask prediction head. The mask head takes feature maps & predicts masks
    # for each region of interest. Like the box predictor, we need to replace the mask head so it can handle
    # the number of classes in the custom dataset.

    # Get the number of input channels for the mask predictor head's final layer
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels  # Number of input channels for the mask prediction head

    # We define the size of the hidden layer in the mask predictor (256 is commonly used)
    hidden_layer = 256

    # Replace the mask predictor with a new one, which is also capable of handling `num_classes`
    # `MaskRCNNPredictor` is a helper class similar to FastRCNNPredictor for the mask head
    # Documentation: https://pytorch.org/vision/stable/_modules/torchvision/models/detection/mask_rcnn.html#MaskRCNNPredictor
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    # At this point, the model is customized to handle a specific number of classes in both the
    # box head & the mask head. The model can now be trained on any dataset with the provided `num_classes`.

    # Return the customized Mask R-CNN model
    return model


In [ ]:
#@title Define Training & Evaluation Functions

'''
This function `train_one_epoch` is designed to train an object detection model (such as Faster R-CNN or Mask R-CNN)
for a single epoch. The function handles multiple essential tasks such as model training,
loss calculation, & optimizer updates. It also logs the loss values during the training process for monitoring.

References:
- PyTorch documentation on training loops: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
- Papers related to object detection:
  - Faster R-CNN: https://arxiv.org/abs/1506.01497
  - Mask R-CNN: https://arxiv.org/abs/1703.06870
'''

def train_one_epoch(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
    epoch: int,
    print_freq: int = 100
    ) -> dict:
    '''
    This function trains an object detection model for one epoch by iterating through batches of data.

    Parameters:
    -----------
    model: torch.nn.Module
        The object detection model (e.g., Mask R-CNN, Faster R-CNN) that will be trained.
    optimizer: torch.optim.Optimizer
        The optimizer used to update the model's weights (e.g., SGD, Adam).
    data_loader: torch.utils.data.DataLoader
        A data loader providing batches of images & targets (ground truth labels & bounding boxes).
    device: torch.device
        The device on which the training will run (typically a GPU or CPU).
    epoch: int
        The current epoch number (for display purposes).
    print_freq: int, optional (default=100)
        How often to print the loss during training (every `print_freq` iterations).

    Returns:
    --------
    metric_logger: dict
        A dictionary storing the total & individual losses accumulated during the epoch.
    '''

    # Set the model to training mode. In this mode, layers like dropout or batch normalization
    # behave differently compared to evaluation mode.
    model.train()

    # Initialize a logger to track & store the loss values during the training process.
    # defaultdict is used here to store the cumulative loss for each loss component.
    metric_logger = defaultdict(float)

    # Iterate over batches of data using the data loader. tqdm is used to display progress in real time.
    # `enumerate` is used to get both the batch index (i) & the data (images, targets).
    for i, (images, targets) in enumerate(tqdm(data_loader, desc=f"Epoch {epoch+1}")):

        # Check if the current batch contains any images. If not, skip this batch.
        if len(images) == 0:
            continue  # Skip if no valid samples

        # Move all images to the specified device (CPU or GPU). Each image is moved individually
        # since `images` is a list of tensors.
        images = [image.to(device) for image in images]

        # Move all target annotations to the specified device. Each target is a dictionary, so
        # we need to move each value within the dictionary to the device.
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass through the model: calculate the losses based on the predictions for the images.
        # The model returns a dictionary where keys are the loss names & values are the loss values.
        loss_dict = model(images, targets)

        # Sum all individual loss values into a total loss.
        losses = sum(loss for loss in loss_dict.values())

        # Get the numeric value of the total loss (converting from a tensor to a float for logging).
        loss_value = losses.item()

        # Zero the gradients before backpropagation to avoid accumulation from previous iterations.
        optimizer.zero_grad()

        # Perform backpropagation to compute gradients of the loss with respect to model parameters.
        losses.backward()

        # Update the model parameters using the optimizer. This applies the gradients computed during backpropagation.
        optimizer.step()

        # Logging: For each loss component in `loss_dict`, accumulate the loss value over the epoch.
        for k, v in loss_dict.items():
            metric_logger[k] += v.item()  # Convert loss tensor to float & accumulate

        # Also accumulate the total loss over the epoch.
        metric_logger['total_loss'] += loss_value

        # Every `print_freq` iterations, print the current iteration number & the total loss.
        if i % print_freq == 0:
            print(f"Iteration {i}, Loss: {loss_value:.4f}")

    # Return the accumulated loss values for analysis or logging outside the function.
    return metric_logger


'''
This function `evaluate` is used to evaluate the performance of an object detection model (like Faster R-CNN or Mask R-CNN)
on a validation or test dataset. It sets the model to evaluation mode & processes batches of images without updating the model weights.
The function uses a metric logger to track the number of batches processed or placeholder metrics. In a real scenario,
you would calculate metrics like mAP (mean Average Precision) or IoU (Intersection over Union) during evaluation.

References:
- PyTorch model evaluation: https://pytorch.org/tutorials/beginner/saving_loading_models.html#save-load-state-dict-recommended
- Object detection evaluation metrics: https://paperswithcode.com/task/object-detection
'''

def evaluate(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device
    ) -> dict:
    '''
    This function evaluates an object detection model on a given dataset without updating the model parameters.
    It processes images & returns evaluation metrics stored in a dictionary.

    Parameters:
    -----------
    model: torch.nn.Module
        The object detection model to evaluate (e.g., Mask R-CNN, Faster R-CNN).
    data_loader: torch.utils.data.DataLoader
        A data loader providing batches of images & their corresponding ground truth labels for evaluation.
    device: torch.device
        The device on which the evaluation will run (typically a GPU or CPU).

    Returns:
    --------
    metric_logger: dict
        A dictionary that stores accumulated metrics from the evaluation process (in this example, only a placeholder).
    '''

    # Set the model to evaluation mode. This disables layers like dropout or batch normalization that behave differently during training.
    # It also ensures that no gradients are calculated during the forward pass.
    model.eval()

    # Initialize a metric logger to store evaluation metrics.
    # In this basic implementation, we only track the number of evaluation steps as a placeholder.
    metric_logger = defaultdict(float)

    # Disable gradient calculations, as we don't need gradients during evaluation.
    # This saves memory & speeds up the process.
    with torch.no_grad():

        # Iterate over batches of images & targets using the data loader.
        # tqdm provides a progress bar for tracking the evaluation process.
        for images, targets in tqdm(data_loader, desc="Evaluating"):

            # Check if the batch contains any images. If there are no valid samples, skip the batch.
            if len(images) == 0:
                continue  # Skip empty batches

            # Move all images to the specified device (CPU or GPU).
            # Each image in the batch is individually transferred to the device.
            images = [img.to(device) for img in images]

            # Move all target annotations (e.g., bounding boxes, labels) to the device as well.
            # Each target is a dictionary, so we need to transfer each value in the dictionary to the device.
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Forward pass through the model to get predictions for the images.
            # The model returns a list of dictionaries, where each dictionary contains the predicted boxes, labels, & scores for an image.
            outputs = model(images)

            # In this example, we're only using a placeholder for evaluation metrics.
            # Normally, you'd calculate metrics such as precision, recall, or mAP here based on the outputs & the ground truth.
            metric_logger['eval'] += 1  # Increment a placeholder metric for each batch processed

    # Return the accumulated metrics after evaluation.
    # In a real scenario, this would include meaningful metrics like mAP or IoU.
    return metric_logger



In [ ]:
#@title Define Visualization Function for Object Detection & Segmentation
'''
This function `visualize_predictions` is used to visualize the predictions made by an object detection or instance segmentation model.
It overlays both ground truth & predicted bounding boxes, masks, & class labels onto the images. This function is essential
for interpreting model outputs during evaluation & debugging.

References:
- PyTorch detection models: https://pytorch.org/vision/stable/models.html
- Object detection & segmentation paper (Mask R-CNN): https://arxiv.org/abs/1703.06870
'''

def visualize_predictions(
    model: torch.nn.Module,
    dataset: torch.utils.data.Dataset,
    device: torch.device,
    num_images: int = 5,
    score_threshold: float = 0.5
    ):
    '''
    This function visualizes predictions for a specified number of images from a dataset by overlaying
    both ground truth & predicted bounding boxes, masks, & labels on the images.

    Parameters:
    -----------
    model: torch.nn.Module
        The object detection or instance segmentation model (e.g., Mask R-CNN, Faster R-CNN) to use for predictions.
    dataset: torch.utils.data.Dataset
        The dataset from which to sample images for visualization.
    device: torch.device
        The device (CPU or GPU) where the model is running.
    num_images: int, optional (default=5)
        The number of images to visualize.
    score_threshold: float, optional (default=0.5)
        The minimum confidence score for displaying predicted boxes & masks.

    Returns:
    --------
    None: Displays visualizations for ground truth & model predictions.
    '''

    # Set the model to evaluation mode. This ensures the model's behavior is appropriate for inference (e.g., no dropout).
    model.eval()

    # Use the category names from the dataset for labeling predictions.
    # This assumes the dataset contains a `COCO_INSTANCE_CATEGORY_NAMES` attribute (used in COCO-format datasets).
    COCO_INSTANCE_CATEGORY_NAMES = dataset.COCO_INSTANCE_CATEGORY_NAMES

    # Get the number of samples in the dataset (total number of images).
    num_samples = len(dataset)

    # Create a list of indices from 0 to num_samples & shuffle them to randomly select images for visualization.
    indices = list(range(num_samples))
    random.shuffle(indices)

    # Generate a color palette for different classes using Seaborn.
    # The palette will assign a unique color to each class.
    num_classes = len(COCO_INSTANCE_CATEGORY_NAMES)
    palette = sns.color_palette('hls', num_classes)
    colors = [palette[i % num_classes] for i in range(num_classes)]

    # Iterate through the randomly selected images (up to `num_images`).
    for idx in range(num_images):

        # Retrieve a sample from the dataset using the randomly selected index.
        sample = dataset[indices[idx]]

        # If the sample is invalid (None), skip to the next iteration.
        if sample is None:
            print(f"Sample {idx} is invalid. Skipping.")
            continue

        # Unpack the sample into image & ground truth target (which contains boxes, masks, labels, etc.).
        img, target = sample

        # Move the image tensor to the specified device (e.g., GPU or CPU).
        img_tensor = img.to(device)

        # Perform inference (model predictions) with no gradients to save memory & computation.
        with torch.no_grad():
            prediction = model([img_tensor])  # The model returns predictions as a list of dictionaries.

        # Convert the image tensor to a NumPy array & unnormalize it (if necessary).
        img_np = img.permute(1, 2, 0).cpu().numpy()  # Convert from (C, H, W) to (H, W, C) format.
        img_np = np.clip(img_np, 0, 1)  # Ensure pixel values are within a valid range (0, 1).

        # Create a figure with three subplots to display the original image, ground truth, & predictions.
        fig, axes = plt.subplots(1, 3, figsize=(9, 2))  # 3 subplots, each 3 inches wide & 2 inches tall.

        # Display the original image in the rightmost subplot.
        axes[2].imshow(img_np)
        axes[2].axis('off')  # Turn off the axis for cleaner visualization.
        axes[2].set_title('Original Image', fontsize=16)

        # Display the image with ground truth annotations (left subplot).
        axes[0].imshow(img_np)
        axes[0].axis('off')
        axes[0].set_title('Ground Truth', fontsize=16)

        # Display the image with predictions (middle subplot).
        axes[1].imshow(img_np)
        axes[1].axis('off')
        axes[1].set_title('Predictions', fontsize=16)

        # Function to draw bounding boxes, masks, & labels on the images.
        def draw_annotations(ax, boxes, masks, labels, scores=None):
            for i, (box, mask, label) in enumerate(zip(boxes, masks, labels)):
                # Convert label index to integer (in case it's a tensor) & handle invalid indices.
                label_idx = label.item() if isinstance(label, torch.Tensor) else label
                if label_idx >= len(COCO_INSTANCE_CATEGORY_NAMES):
                    print(f"Label index {label_idx} out of range for COCO_INSTANCE_CATEGORY_NAMES.")
                    continue

                # Get the color for this class from the color palette.
                color = colors[label_idx]
                color_rgb = to_rgb(color)  # Convert the color to RGB format for matplotlib.

                # Extract the bounding box coordinates (x1, y1, x2, y2).
                x1, y1, x2, y2 = box.cpu().numpy() if isinstance(box, torch.Tensor) else box
                width, height = x2 - x1, y2 - y1

                # Draw the bounding box as a rectangle.
                rect = patches.Rectangle(
                    (x1, y1), width, height, linewidth=2,
                    edgecolor=color, facecolor='none',  # Bounding box color.
                    linestyle='-', alpha=0.7
                )
                ax.add_patch(rect)  # Add the rectangle to the plot.

                # Convert the mask tensor to a binary mask (1 for the object, 0 for the background).
                mask = mask.cpu().numpy() if isinstance(mask, torch.Tensor) else mask
                mask = mask[0] if mask.ndim == 3 else mask  # Handle cases where the mask has extra dimensions.
                mask = mask > 0.5  # Binarize the mask.

                # Create an RGBA mask for displaying (with transparency).
                rgba_mask = np.zeros((mask.shape[0], mask.shape[1], 4))
                for c in range(3):
                    rgba_mask[:, :, c] = color_rgb[c]  # Assign the RGB color.
                rgba_mask[:, :, 3] = mask * 0.5  # Set the alpha (transparency) channel.

                # Overlay the mask on the image.
                ax.imshow(rgba_mask)

                # Add the class label & confidence score (if available) as text above the bounding box.
                class_name = COCO_INSTANCE_CATEGORY_NAMES[label_idx]
                score_text = f": {scores[i]:.2f}" if scores is not None else ""  # Show score if provided.
                text = f"{class_name}{score_text}"
                ax.text(
                    x1, y1 - 5, text,
                    fontsize=10, color='white',
                    bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', pad=2),  # Text box styling.
                    verticalalignment='top'
                )

        # Draw ground truth annotations (boxes, masks, labels).
        draw_annotations(
            axes[0],
            target['boxes'],
            target['masks'],
            target['labels']
        )

        # Filter predicted boxes by score threshold (only keep high-confidence predictions).
        pred_boxes = prediction[0]['boxes'].cpu()
        pred_masks = prediction[0]['masks'].cpu()
        pred_scores = prediction[0]['scores'].cpu()
        pred_labels = prediction[0]['labels'].cpu()

        # Filter predictions by score threshold.
        high_score_idxs = pred_scores > score_threshold
        pred_boxes = pred_boxes[high_score_idxs]
        pred_masks = pred_masks[high_score_idxs]
        pred_scores = pred_scores[high_score_idxs]
        pred_labels = pred_labels[high_score_idxs]

        # Draw predicted annotations (boxes, masks, labels).
        draw_annotations(
            axes[1],
            pred_boxes,
            pred_masks,
            pred_labels,
            scores=pred_scores
        )

        # Adjust the layout of the figure & display the visualizations.
        plt.tight_layout()
        plt.show()


In [ ]:
#@title Main Run Script for Object Detection & Instance Segmentation
'''
This script demonstrates the full pipeline for training, evaluating, & visualizing predictions using a Mask R-CNN model.
It includes steps for loading the COCO dataset, setting up the model & training, running evaluation, & visualizing results.
The code is structured to run on Google Colab with Google Drive mounted to store & retrieve data & model checkpoints.

References:
- Mask R-CNN paper: https://arxiv.org/abs/1703.06870
- COCO dataset: https://cocodataset.org/
- PyTorch detection models: https://pytorch.org/vision/stable/models.html
'''

# Mount Google Drive to access datasets & save models directly to your drive
from google.colab import drive
drive.mount('/content/drive')

# Determine whether to use a GPU (if available) or fallback to CPU for model training & inference.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

# Paths to the COCO dataset on Google Drive
# The data_dir contains the COCO dataset. Update this path based on your setup.
data_dir = '/content/drive/MyDrive/data_535743/coco2017_reduced_1000/'  # Ensure this path is correct.
train_images = os.path.join(data_dir, 'train2017')  # Path to training images
train_annotations = os.path.join(data_dir, 'annotations', 'instances_train2017.json')  # Training annotations
val_images = os.path.join(data_dir, 'val2017')  # Path to validation images
val_annotations = os.path.join(data_dir, 'annotations', 'instances_val2017.json')  # Validation annotations

# Verify that both the training & validation annotation files exist before proceeding.
# This ensures the dataset paths are correctly set up.
assert os.path.exists(train_annotations), f"Train annotation file not found: {train_annotations}"
assert os.path.exists(val_annotations), f"Validation annotation file not found: {val_annotations}"

# Initialize the custom COCO dataset for training & validation, applying necessary transformations.
# `transform` is typically a series of data augmentation steps (e.g., resizing, normalization).
dataset_train = COCODataset(root=train_images, annFile=train_annotations, transforms=transform)
dataset_val = COCODataset(root=val_images, annFile=val_annotations, transforms=transform)

# Determine the number of object classes by checking the number of category IDs in the dataset.
# Add 1 for the background class (usually class 0 in object detection models).
num_classes = len(dataset_train.category_ids) + 1  # +1 for the background class

# Number of worker processes for loading data in parallel. Set to 0 for environments like Google Colab.
num_workers = 0  # Reduce workers in Colab to avoid issues.

# Initialize the data loader for training, which loads batches of images & annotations (targets).
# The collate_fn function helps batch data properly when some samples have different shapes.
train_loader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=4,  # Adjust based on available GPU memory
    shuffle=True,  # Shuffle the dataset to ensure the model sees varied samples in each epoch
    num_workers=num_workers,
    collate_fn=collate_fn  # Custom function to handle batching for variable-sized tensors
)

# Initialize the data loader for validation. This data loader does not shuffle the data.
val_loader = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=4,  # Batch size for validation
    shuffle=False,  # No need to shuffle for validation
    num_workers=num_workers,
    collate_fn=collate_fn  # Custom batching function
)

# Initialize the Mask R-CNN model with the correct number of classes.
# The function `get_instance_segmentation_model` customizes the model for the current task.
model = get_instance_segmentation_model(num_classes)

# Move the model to the device (GPU or CPU) for training & inference.
model.to(device)

# Define the optimizer (Stochastic Gradient Descent) for updating model weights during training.
# It uses momentum to accelerate learning & weight decay to prevent overfitting.
params = [p for p in model.parameters() if p.requires_grad]  # Trainable parameters
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Define a learning rate scheduler that reduces the learning rate every `step_size` epochs.
# Here, the learning rate is reduced by a factor of `gamma` every 3 epochs.
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Set the number of epochs for training. Adjust this number based on available resources & time.
num_epochs = 3  # Run for 3 epochs; increase for better performance with larger datasets.

# Training loop that runs for the specified number of epochs.
for epoch in range(num_epochs):
    print(f"Starting Epoch {epoch+1}/{num_epochs}")

    # Train the model for one epoch & return metrics (like loss) for monitoring.
    # The `train_one_epoch` function handles the full training loop for a single epoch.
    train_metrics = train_one_epoch(model, optimizer, train_loader, device, epoch, print_freq=100)

    # Step the learning rate scheduler at the end of each epoch to adjust the learning rate.
    lr_scheduler.step()

    # Evaluate the model on the validation dataset. `evaluate` runs inference & calculates metrics.
    val_metrics = evaluate(model, val_loader, device)

    # Print the loss after each epoch for tracking progress.
    print(f"Epoch {epoch+1} completed. Train Loss: {train_metrics['total_loss']:.4f}")

# Save the trained model's state (weights) to a file on Google Drive.
# This allows you to load & use the model later without retraining it.
model_save_path = '/content/drive/MyDrive/maskrcnn_instance_segmentation.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved as {model_save_path}")

# Visualize the predictions made by the trained model on the validation dataset.
# Displays a few images with predicted bounding boxes, masks, & labels.
visualize_predictions(model, dataset_val, device, num_images=3, score_threshold=0.5)

## **4. DETR Experiment**

In [ ]:
#@title Install Necessary Libraries & Restart the Session
'''
This block of code installs essential libraries (`transformers` & `timm`) for working with
state-of-the-art models in computer vision & natural language processing. After installing the libraries,
the session is restarted to ensure that the libraries are loaded correctly into the environment.

- `transformers`: A popular library by Hugging Face that provides pre-trained models for NLP tasks & vision-language models.
  Documentation: https://huggingface.co/transformers/

- `timm`: A library by Ross Wightman providing a large collection of pre-trained models for image classification, segmentation, etc.
  Documentation: https://github.com/rwightman/pytorch-image-models

This script is specifically designed for environments like Google Colab or Jupyter Notebooks, where you can dynamically install packages & restart the runtime to ensure proper initialization of installed packages.

References:
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- PyTorch Image Models (timm): https://rwightman.github.io/pytorch-image-models/
'''

# Install the required libraries using the `pip` package manager.
# - `transformers` for working with pre-trained models from Hugging Face.
# - `timm` (PyTorch Image Models) for accessing a wide range of pre-trained image models.
!pip install transformers timm

# Import the time module to add a delay before restarting the session.
import time

# Import `clear_output` from IPython to clear the notebook output, ensuring a clean display for the user.
from IPython.display import clear_output

# Clear the output after the packages are installed to make the notebook cleaner.
clear_output()

# Print a message to let the user know that the libraries are installed & the session will restart.
print("Necessary Libraries are Installed. Restarting the session!")

# Add a short delay (1 second) before restarting to allow the message to be displayed to the user.
time.sleep(1)

# Import the `os` module to access low-level operating system functionality.
import os

# Use `os._exit(00)` to exit the current Python runtime environment forcefully.
# This effectively simulates a restart in notebook environments like Google Colab or Jupyter.
# After this command, the environment will be restarted & all the packages installed will be properly loaded.
os._exit(00)

In [ ]:
#@title Import Libraries for Object Detection & Segmentation with DEtection TRansformers (DETR)
'''
This block of code imports essential libraries for working with an object detection & segmentation
model called DEtection TRansformers (DETR). DETR is a powerful model architecture that uses Transformers
to perform object detection & segmentation tasks in an end-to-end manner without the need for traditional
hand-crafted features like anchor boxes.

References:
- DETR Paper: https://arxiv.org/abs/2005.12872
- Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers/
'''

# Import PyTorch, the deep learning framework used for creating models, training, & making predictions.
# PyTorch provides a flexible & efficient way to work with tensors, the core data structure for deep learning.
import torch

# Import `DetrImageProcessor` & `DetrForSegmentation` from the Hugging Face Transformers library.
# These are specialized tools designed to preprocess images & run DETR models for segmentation tasks.
# DetrImageProcessor: Processes & normalizes images to feed them into the model.
# DetrForSegmentation: A pre-trained DETR model that performs both object detection & segmentation.
from transformers import DetrImageProcessor, DetrForSegmentation

# Import the `requests` library, which allows us to make HTTP requests.
# In this case, it will be used to download images from the web for processing.
import requests

# Import `Image` from the Python Imaging Library (PIL) to open, manipulate, & display image data.
# PIL (or its modern equivalent, Pillow) is commonly used for handling images in Python.
from PIL import Image

# Import `matplotlib.pyplot`, a widely-used library for creating visualizations like plots, charts, & image displays.
# It allows us to visualize images, bounding boxes, & segmentation masks.
import matplotlib.pyplot as plt

# Import `patches` from `matplotlib` to draw shapes like rectangles on images.
# In object detection, patches can be used to display bounding boxes around detected objects.
import matplotlib.patches as mpatches

# Import NumPy, a powerful library for numerical computations in Python.
# It is commonly used in conjunction with PyTorch for manipulating arrays & image data.
import numpy as np

In [ ]:
#@title Main Run for Object Detection & Segmentation with DEtection TRansformers (DETR)
'''
This script demonstrates how to use a pre-trained DEtection TRansformer (DETR) model for panoptic segmentation.
Panoptic segmentation is a task that involves both object detection (assigning a class label to each object)
and semantic segmentation (assigning a class label to each pixel).

The steps include loading the model, preparing the image, making predictions, & visualizing the results.

References:
- DETR (DEtection TRansformers) paper: https://arxiv.org/abs/2005.12872
- Panoptic Segmentation: https://arxiv.org/abs/1801.00868
- Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers/
'''

# Importing Google Colab's drive module to mount Google Drive, where datasets or model checkpoints might be stored.
from google.colab import drive
# Mount Google Drive at the specified path '/content/drive', so files in the Drive can be accessed in this Colab session.
drive.mount('/content/drive')

# Load the pre-trained DETR model & the associated image processor.
# - `DetrImageProcessor` handles image preprocessing (e.g., resizing, normalization) for the model.
# - `DetrForSegmentation` is a pre-trained DETR model specialized for panoptic segmentation, which combines instance segmentation & semantic segmentation tasks.
processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50-panoptic')

# Detect whether a GPU (CUDA) is available, as using a GPU significantly speeds up the model's inference process.
# If CUDA is not available, it defaults to using the CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the pre-trained model `DetrForSegmentation` for panoptic segmentation & move it to the specified device (GPU or CPU).
model = DetrForSegmentation.from_pretrained('facebook/detr-resnet-50-panoptic').to(device)

# Download a sample image from the COCO dataset using a URL.
# The COCO dataset (Common Objects in Context) is a widely-used dataset for tasks like object detection & segmentation.
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'

# Use `requests.get` to fetch the image from the web, then open it with the Python Imaging Library (PIL).
# The image is converted to RGB format to ensure compatibility with the model (which expects 3-channel color images).
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# Prepare the image for the model using the processor.
# The processor takes care of resizing, normalizing, & converting the image into the format expected by the model.
# `return_tensors="pt"` indicates that the output will be a PyTorch tensor.
inputs = processor(images=image, return_tensors="pt").to('cuda' if torch.cuda.is_available() else 'cpu')

# Perform a forward pass through the model to get the outputs.
# `torch.no_grad()` is used to disable gradient calculation, as this is inference, not training, which saves memory & speeds up processing.
with torch.no_grad():
    outputs = model(**inputs)

# Post-process the model's output to obtain the panoptic segmentation results.
# The `post_process_panoptic_segmentation` method converts the raw outputs from the model into a human-readable segmentation map.
# This map contains information about both the segments (regions) & the objects in the image.
result = processor.post_process_panoptic_segmentation(outputs)[0]

# Extract the segmentation map & the corresponding category IDs for each segment.
# The segmentation map is a 2D array where each pixel's value corresponds to a segment ID.
segmentation_map = result['segmentation'].cpu().numpy()  # Move the result to CPU & convert to a NumPy array.
segments_info = result['segments_info']  # Metadata for each segment, including category IDs & object IDs.

# Create a color palette for visualization by generating random RGB values for each label.
# The number of labels (categories) in the model's configuration defines the number of unique colors needed.
palette = np.random.randint(0, 255, (model.config.num_labels, 3), dtype=np.uint8)

# Use the segmentation map to create a colorized version of the segmentation result.
# Each segment is assigned a color from the palette based on its segment ID.
seg_image = palette[segmentation_map]

In [ ]:
#@title Display the Original Image & the Panoptic Segmentation
'''
This code visualizes the original input image alongside the model's panoptic segmentation output.
Panoptic segmentation combines both instance & semantic segmentation, identifying not just object instances
but also background classes like "sky" or "road." The visualization includes a global legend that shows the
corresponding class names & colors for each segment in the image.

References:
- Matplotlib documentation for plotting: https://matplotlib.org/stable/contents.html
- Panoptic Segmentation paper: https://arxiv.org/abs/1801.00868
'''

# Create a figure with two subplots using `plt.subplots`.
# The figsize parameter sets the overall size of the figure (10 inches wide by 5 inches tall).
# One subplot will display the original image, & the other will show the segmentation output.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Display the original image on the first subplot (`ax1`).
# The image is displayed using `imshow`, & `axis('off')` hides the axis ticks & labels for cleaner presentation.
ax1.imshow(image)
ax1.axis('off')  # Turn off axis for cleaner visualization
ax1.set_title('Original Image', fontsize=16)  # Set title & adjust font size

# Display the panoptic segmentation map on the second subplot (`ax2`).
# `seg_image` is a colorized version of the segmentation output generated from the palette.
ax2.imshow(seg_image)
ax2.axis('off')  # Turn off axis for cleaner visualization
ax2.set_title('Panoptic Segmentation', fontsize=16)  # Set title & adjust font size

# Extract the class names & color palette used by the model.
# `id2label` is a dictionary that maps class IDs to human-readable class names (e.g., "car", "person").
id2label = model.config.id2label  # Retrieve class labels from model configuration

# Create a list of class categories present in the image based on the `segments_info` from the segmentation output.
# Each item in `segments_info` contains metadata about the detected segments, including the class (label) ID.
categories = [info['label_id'] for info in segments_info]

# Create a list of labels for the legend. If a category ID is not found in `id2label`, use 'Unknown' as a fallback label.
# The first label is 'unknown' (it represents no specific category).
legend_labels = ['unknown'] + [id2label.get(category, 'Unknown') for category in categories]

# Create a list of `Patch` objects to display in the legend.
# Each patch represents a color (from the palette) corresponding to a class label.
# `palette[i] / 255` converts the RGB values from 0-255 to 0-1 (as required by Matplotlib).
legend_elements = [
    mpatches.Patch(facecolor=np.array(palette[i]) / 255, label=legend_label)  # Create a patch for each class
    for legend_label, i in zip(legend_labels, np.unique(segmentation_map))  # Zip labels & unique class IDs from the segmentation map
]

# Add a global legend to the figure, displaying the class names & their associated colors.
# `fig.legend` adds the legend to the entire figure.
# `bbox_to_anchor` positions the legend below the subplots, `ncol=3` arranges the legend in 3 columns, & `fontsize=16` adjusts the font size.
fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=3, fontsize=16)

# Display the figure with both subplots & the legend.
plt.show()

## **5. SegFormer  Experiment**

In [ ]:
# @title
#@title Install Necessary Libraries & Restart the Session
'''
This block of code installs essential libraries (`transformers` & `timm`) for working with
state-of-the-art models in computer vision & natural language processing. After installing the libraries,
the session is restarted to ensure that the libraries are loaded correctly into the environment.

- `transformers`: A popular library by Hugging Face that provides pre-trained models for NLP tasks & vision-language models.
  Documentation: https://huggingface.co/transformers/

- `timm`: A library by Ross Wightman providing a large collection of pre-trained models for image classification, segmentation, etc.
  Documentation: https://github.com/rwightman/pytorch-image-models

This script is specifically designed for environments like Google Colab or Jupyter Notebooks, where you can dynamically install packages & restart the runtime to ensure proper initialization of installed packages.

References:
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- PyTorch Image Models (timm): https://rwightman.github.io/pytorch-image-models/
'''

# Install the required libraries using the `pip` package manager.
# - `transformers` for working with pre-trained models from Hugging Face.
# - `timm` (PyTorch Image Models) for accessing a wide range of pre-trained image models.
!pip install transformers timm

# Import the time module to add a delay before restarting the session.
import time

# Import `clear_output` from IPython to clear the notebook output, ensuring a clean display for the user.
from IPython.display import clear_output

# Clear the output after the packages are installed to make the notebook cleaner.
clear_output()

# Print a message to let the user know that the libraries are installed & the session will restart.
print("Necessary Libraries are Installed. Restarting the session!")

# Add a short delay (1 second) before restarting to allow the message to be displayed to the user.
time.sleep(1)

# Import the `os` module to access low-level operating system functionality.
import os

# Use `os._exit(00)` to exit the current Python runtime environment forcefully.
# This effectively simulates a restart in notebook environments like Google Colab or Jupyter.
# After this command, the environment will be restarted & all the packages installed will be properly loaded.
os._exit(00)

In [ ]:
# @title
#@title Import Libraries for Semantic Segmentation with SegFormer
'''
This code imports essential libraries for performing semantic segmentation using the SegFormer model.
SegFormer is a Transformer-based model designed for efficient & accurate semantic segmentation tasks.
Semantic segmentation involves classifying each pixel in an image into a specific category (e.g., road, car, building).

References:
- SegFormer Paper: https://arxiv.org/abs/2105.15203
- Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers/
'''

# Import the `torch` library, which is the core of the PyTorch framework.
# PyTorch provides tools for building & training deep learning models, & handling tensor operations.
import torch

# Import the SegFormer-related classes from Hugging Face's `transformers` library.
# `SegformerImageProcessor`: This is used to preprocess images to be fed into the SegFormer model.
# It handles tasks like resizing, normalization, & conversion into tensors.
# `SegformerForSemanticSegmentation`: This class contains the pre-trained SegFormer model for performing semantic segmentation.
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

# Import the `Image` class from the Python Imaging Library (PIL) to open & manipulate image files.
# PIL (or its modern equivalent, Pillow) allows us to load images & convert them into different formats or manipulate pixel values.
from PIL import Image

# Import `matplotlib.pyplot`, a popular library used for creating visualizations such as plots & charts.
# Here, `plt` will be used to display images & segmentation results.
import matplotlib.pyplot as plt

# Import `patches` from `matplotlib` to add annotations like bounding boxes or patches for drawing on images.
# Although this script focuses on segmentation, patches could be useful for additional visualizations like marking specific areas in the image.
import matplotlib.patches as mpatches

# Import `numpy`, a powerful library for numerical computing.
# NumPy is used here for manipulating image arrays, particularly when handling segmentation maps (which are often represented as arrays).
import numpy as np

In [ ]:
# @title
#@title Main Run for Semantic Segmentation using SegFormer
'''
This script demonstrates how to use a pre-trained SegFormer model for semantic segmentation.
It loads a pre-trained model & image processor from Hugging Face, processes an input image,
performs inference to get the segmentation map, & post-processes the result for visualization.

References:
- SegFormer Paper: https://arxiv.org/abs/2105.15203
- ADE20K Dataset (used in pre-trained model): https://groups.csail.mit.edu/vision/datasets/ADE20K/
- Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers/
'''

# Import Google Drive for accessing files stored in the cloud.
# This is useful in environments like Google Colab where files can be stored in the user's Google Drive.
from google.colab import drive

# Mount Google Drive to the /content/drive directory so that files (e.g., datasets, models)
# stored on Google Drive can be accessed from the notebook.
drive.mount('/content/drive')

# Determine whether to run on a GPU (CUDA) or fall back to CPU.
# Using a GPU significantly speeds up the model's inference time, especially with deep learning models.
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load a pre-trained SegFormer model & image processor from the Hugging Face model hub.
# - `SegformerImageProcessor`: Handles pre-processing of images (e.g., resizing, normalization) to make them compatible with the model.
# - `SegformerForSemanticSegmentation`: The pre-trained SegFormer model, which is fine-tuned on the ADE20K dataset (a common dataset for semantic segmentation tasks).
# The specific model used here is `segformer-b0`, a smaller version of SegFormer that is efficient but still powerful.
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")

# Load the SegFormer model, which has been fine-tuned for semantic segmentation tasks on ADE20K.
# The model is loaded onto the specified device (GPU or CPU).
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512"  # Pre-trained model checkpoint
).to(device)

# Download a sample image or load an image from Google Drive for segmentation.
# The image is opened using the PIL (Python Imaging Library) & converted to RGB format to ensure it's compatible with the model.
# Here, the image is loaded from the user's Google Drive.
image = Image.open("/content/drive/MyDrive/data_535743/coco2017_reduced_1000/val2017/000000458663.jpg").convert("RGB")

# Preprocess the image using the SegformerImageProcessor to get it into a format suitable for the model.
# The processor handles tasks like resizing the image & converting it to a PyTorch tensor.
# `return_tensors="pt"` returns a tensor in PyTorch format. The image is moved to the correct device (GPU or CPU).
inputs = processor(images=image, return_tensors="pt").to(device)

# Perform inference (a forward pass through the model) to get the model's predictions.
# `torch.no_grad()` ensures that gradients are not calculated, which saves memory & computation time since we're only doing inference.
with torch.no_grad():
    outputs = model(**inputs)  # Outputs will contain the raw predictions (logits) from the model

# Post-process the outputs from the model to get a human-readable segmentation map.
# `post_process_semantic_segmentation` converts the raw model outputs (logits) into a segmentation map,
# where each pixel is assigned a class label.
# `target_sizes=[image.size[::-1]]` ensures the output segmentation map matches the input image's size (height, width).
segmentation = processor.post_process_semantic_segmentation(
    outputs, target_sizes=[image.size[::-1]]
)[0]  # Get the first (and only) output for this single image

# The segmentation map is a tensor of shape [height, width], where each pixel's value represents its class label.
# Convert the segmentation tensor from PyTorch format to a NumPy array, which is more convenient for further processing & visualization.
pred_seg = segmentation.cpu().numpy()  # Move the tensor to CPU & convert it to a NumPy array for visualization

In [ ]:
# @title
#@title Display the Original Image & Semantic Segmentation Output
'''
This code visualizes the original input image & the corresponding semantic segmentation output from a pre-trained model.
It creates a side-by-side comparison of the original image & the segmentation result, with a color-coded legend
to indicate which class each color represents.

References:
- ADE20K Dataset (used in this model): https://groups.csail.mit.edu/vision/datasets/ADE20K/
- Matplotlib documentation: https://matplotlib.org/stable/contents.html
'''

# Get the class names (id2label) from the model configuration
# id2label is a dictionary mapping class IDs to human-readable class names (e.g., 0: "background", 1: "wall", etc.)
id2label = model.config.id2label

# Convert `id2label` to a list, sorted by key (class ID) for better accessibility.
# ADE20K has 150 classes, so we create a list of class names where the index represents the class ID.
ADE20K_CLASSES = [id2label[i] for i in range(len(id2label))]

# Create a random color palette for the ADE20K classes.
# The palette is an array of shape [150, 3] where each class is assigned a random RGB color.
# `model.config.num_labels` refers to the number of classes in the model (150 for ADE20K).
ade_palette = np.random.randint(0, 255, (model.config.num_labels, 3), dtype=np.uint8)

# Convert the generated color palette to a NumPy array for easier manipulation & consistent data type.
palette = np.array(ade_palette, dtype=np.uint8)

# Use the color palette to create a colorized version of the segmentation map (`pred_seg`).
# `pred_seg` is an array where each pixel value represents a class ID (0-149).
# The palette is applied to map each class ID to its corresponding color in the `palette` array.
segmentation_image = palette[pred_seg]

# Create a figure with two subplots to display the original image & the segmentation result side by side.
# figsize=(15, 7) sets the size of the entire figure (15 inches wide & 7 inches tall).
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

# Display the original image in the first subplot (ax1).
# `axis('off')` hides the axis & ticks to provide a cleaner display of the image.
ax1.imshow(image)
ax1.axis('off')
ax1.set_title('Original Image', fontsize=14)  # Set the title with a font size of 14.

# Display the segmentation image in the second subplot (ax2).
# This image is the result of applying the color palette to the predicted segmentation.
ax2.imshow(segmentation_image)
ax2.axis('off')  # Hide axis & ticks for a cleaner presentation.
ax2.set_title('Semantic Segmentation', fontsize=14)  # Set the title for this subplot.

# Create a legend that maps class colors to their corresponding names.
# `mpatches.Patch` creates a colored patch for each class in the segmentation map.
# `np.unique(pred_seg)` returns the unique class IDs present in the predicted segmentation, ensuring that only classes
# present in the image are displayed in the legend.
legend_elements = [
    mpatches.Patch(facecolor=np.array(ade_palette[cls]) / 255, label=id2label.get(cls, 'Unknown'))  # Normalize color to 0-1 range
    for cls in np.unique(pred_seg)  # Loop over unique class IDs in the predicted segmentation map
]

# Add the legend to the figure.
# The legend is placed at the bottom of the figure (`loc='lower center'`) with the `bbox_to_anchor` argument to adjust positioning.
# `ncol=4` arranges the legend in 4 columns, & `fontsize=13` sets the font size for the class names.
fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.05), ncol=4, fontsize=13)

# Display the figure with the original image, segmentation output, & legend.
plt.show()

## **6. Pose Estimation**

In [ ]:
# @title
#@title Install Necessary Libraries & Restart the Session
'''
This script installs the required libraries for performing tasks such as object detection, segmentation,
or other computer vision tasks using MediaPipe. After installing, the script restarts the session to ensure
that the newly installed libraries are properly loaded into the runtime environment. This approach is particularly useful
in environments like Google Colab or Jupyter notebooks.

References:
- MediaPipe Documentation: https://google.github.io/mediapipe/
'''

# The first line installs the `mediapipe` library using the `pip` package manager.
# MediaPipe is a cross-platform framework by Google that provides a wide variety of computer vision & machine learning solutions.
# It includes ready-to-use models for tasks like hand tracking, face detection, pose estimation, etc.
!pip install mediapipe

# The `time` library is imported to introduce delays or pauses during script execution.
# This is useful when adding a delay between tasks, such as waiting before restarting the session.
import time

# The `IPython.display` module provides tools for controlling the display of output in Jupyter notebooks.
# Here, we import `clear_output` to clear the current output, ensuring that the notebook is refreshed & clean after installation.
from IPython.display import clear_output

# After installing the necessary libraries, the current output is cleared to prevent clutter.
# This removes the installation logs from the notebook display, providing a cleaner user interface.
clear_output()

# Print a message indicating that the required libraries have been installed.
# This message informs the user that the environment will now restart to load the installed libraries properly.
print("Necessary Libraries are Installed. Restarting the session!")

# Introduce a delay of 1 second using `time.sleep(1)`.
# This ensures that the "Restarting the session!" message is visible for at least one second before the session is restarted.
time.sleep(1)

# The `os` module is imported to perform operating system tasks such as interacting with the file system or manipulating the environment.
import os

# The `os._exit(00)` command forcibly exits the current Python session with the exit code 0, indicating successful completion.
# This command is necessary in environments like Google Colab or Jupyter notebooks, where a session restart is required after package installation.
# When the environment is restarted, the installed libraries will be loaded correctly.
os._exit(00)

In [ ]:
# @title
#@title Import Necessary Libraries for Computer Vision & Pose Estimation
'''
This code imports the necessary libraries for performing computer vision tasks, specifically using MediaPipe for pose estimation,
along with tools for image processing & visualization. Each library serves a specific purpose in the pipeline, such as loading images,
processing them, & visualizing the output.

References:
- OpenCV Documentation: https://docs.opencv.org/
- MediaPipe Pose Documentation: https://google.github.io/mediapipe/solutions/pose.html
- Matplotlib Documentation: https://matplotlib.org/stable/contents.html
- NumPy Documentation: https://numpy.org/doc/
- PIL (Pillow) Documentation: https://pillow.readthedocs.io/
'''

# Import OpenCV (cv2) for computer vision tasks.
# OpenCV is a powerful library for image processing, computer vision, & machine learning tasks.
# It provides functions to load, manipulate, & analyze images & video streams.
import cv2

# Import MediaPipe for pose estimation & other machine learning solutions.
# MediaPipe is a cross-platform framework by Google that provides ready-to-use machine learning models for tasks such as pose estimation,
# hand tracking, face detection, & more.
import mediapipe as mp

# Import Matplotlib's pyplot module for visualizing images, plots, & data.
# Matplotlib is widely used in Python for creating static, animated, & interactive visualizations.
import matplotlib.pyplot as plt

# Import NumPy, a powerful library for numerical computations.
# NumPy is often used in image processing to manipulate image arrays (as images are represented as multi-dimensional arrays).
import numpy as np

# Import Image from the Python Imaging Library (PIL), specifically the Pillow library.
# PIL (or Pillow) allows easy image manipulation, such as opening, resizing, or converting between formats.
# It is often used for loading & saving images in various formats.
from PIL import Image

In [ ]:
# @title
#@title Main Run for Pose Estimation using MediaPipe
'''
This code demonstrates how to use the MediaPipe library for performing human pose estimation.
The MediaPipe Pose module detects human body landmarks (keypoints) from an input image & draws the detected
landmarks & pose connections on the image. This is particularly useful for applications such as sports analysis,
motion capture, & fitness tracking.

References:
- MediaPipe Pose Documentation: https://google.github.io/mediapipe/solutions/pose.html
- MediaPipe Drawing Utils: https://google.github.io/mediapipe/solutions/drawing_styles.html
'''

# Import Google Drive integration from Colab to access files stored in the user's Google Drive.
# This allows us to load data such as images stored in the cloud directly into the Colab environment.
from google.colab import drive

# Mount Google Drive at the `/content/drive` path so that files from the user's Google Drive are accessible in this Colab session.
# After mounting, files like images or datasets stored in Google Drive can be loaded & used in the notebook.
drive.mount('/content/drive')

# Initialize the MediaPipe Pose class, which is used to perform pose estimation.
# MediaPipe Pose is a machine learning model that detects human body landmarks from an image.
# `mp_pose` is the module that contains all the pose estimation utilities.
mp_pose = mp.solutions.pose

# Create an instance of the `Pose` class to detect body landmarks from an image.
# `static_image_mode=True` ensures the model treats each image as independent, meaning it doesn't expect frames in a video sequence.
# This mode is suited for processing single images where the model resets after processing each image.
pose = mp_pose.Pose(static_image_mode=True)

# Load an image from the user's Google Drive using the Python Imaging Library (PIL).
# The image is located at the specified path in the Google Drive.
# The `convert("RGB")` function ensures the image is in RGB format, which is required for many models.
image = Image.open("/content/drive/MyDrive/data_535743/coco2017_reduced_1000/val2017/000000362716.jpg").convert("RGB")

# Convert the loaded image into a NumPy array. This format is necessary for MediaPipe as it processes images in array form.
# NumPy arrays are commonly used in image processing tasks because they allow efficient manipulation of pixel values.
image = np.array(image)

# Perform pose estimation on the input image using the MediaPipe Pose model.
# The `pose.process(image)` function detects the body landmarks in the image.
# The result of this operation contains detected landmarks (if any) stored in the `results` object.
results = pose.process(image)

# Initialize MediaPipe's drawing utilities for visualizing the detected landmarks.
# `mp_drawing` provides tools to draw landmarks & connections (e.g., lines connecting keypoints) on the image.
mp_drawing = mp.solutions.drawing_utils

# Create a copy of the original image for drawing the pose landmarks.
# The landmarks will be drawn on this `annotated_image`, leaving the original image unmodified.
annotated_image = image.copy()

# Check if any pose landmarks were detected in the image.
# If landmarks are detected (i.e., `results.pose_landmarks` is not None), draw the landmarks & connections on the image.
if results.pose_landmarks:
    # Use `draw_landmarks` to draw the detected pose landmarks on the `annotated_image`.
    # `results.pose_landmarks` contains the coordinates of detected landmarks.
    # `mp_pose.POSE_CONNECTIONS` is a predefined set of landmark connections (e.g., lines connecting joints like shoulder to elbow).
    mp_drawing.draw_landmarks(
        annotated_image,  # The image on which to draw the landmarks
        results.pose_landmarks,  # Detected pose landmarks
        mp_pose.POSE_CONNECTIONS  # Connections between landmarks (e.g., connecting joints)
    )

In [ ]:
# @title
#@title Display the Original & Annotated Images for Pose Estimation
'''
This code uses Matplotlib to display two images side-by-side: the original image & the annotated image
with pose estimation results. The pose estimation annotations (landmarks) are drawn on the second image
using a model like MediaPipe Pose. This type of visualization helps in comparing the original input
with the model's predictions.

References:
- Matplotlib Documentation: https://matplotlib.org/stable/contents.html
- MediaPipe Pose Documentation: https://google.github.io/mediapipe/solutions/pose.html
'''

# Create a figure with two subplots arranged horizontally using `plt.subplots`.
# figsize=(10, 5) specifies the size of the entire figure (10 inches wide, 5 inches tall).
# The two subplots will display the original image & the annotated image with pose estimations.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Display the original image on the first subplot (`ax1`).
# `imshow(image)` renders the original image, which was loaded or processed earlier in the code.
ax1.imshow(image)

# Turn off the axis labels & ticks for the first subplot to make the visualization cleaner.
# `axis('off')` ensures no unnecessary axis details are displayed.
ax1.axis('off')

# Set the title for the first subplot to 'Original Image'.
# The title helps differentiate between the original & annotated images in the visualization.
ax1.set_title('Original Image')

# Display the annotated image (with pose landmarks) on the second subplot (`ax2`).
# `annotated_image` is a copy of the original image with pose landmarks drawn on it (from previous steps).
ax2.imshow(annotated_image)

# Turn off the axis labels & ticks for the second subplot to keep the visualization focused on the image.
ax2.axis('off')

# Set the title for the second subplot to 'Pose Estimation', indicating that it contains the pose landmarks.
ax2.set_title('Pose Estimation')

# Display the figure containing both subplots.
# `plt.show()` renders the figure with the two images side-by-side in the notebook or output window.
plt.show()

## **7. DPTForDepthEstimation**

In [ ]:
#@title Install Necessary Libraries & Restart the Session
'''
This block of code installs essential libraries (`transformers` & `timm`) for working with
state-of-the-art models in computer vision & natural language processing. After installing the libraries,
the session is restarted to ensure that the libraries are loaded correctly into the environment.

- `transformers`: A popular library by Hugging Face that provides pre-trained models for NLP tasks & vision-language models.
  Documentation: https://huggingface.co/transformers/

- `timm`: A library by Ross Wightman providing a large collection of pre-trained models for image classification, segmentation, etc.
  Documentation: https://github.com/rwightman/pytorch-image-models

This script is specifically designed for environments like Google Colab or Jupyter Notebooks, where you can dynamically install packages & restart the runtime to ensure proper initialization of installed packages.

References:
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- PyTorch Image Models (timm): https://rwightman.github.io/pytorch-image-models/
'''

# Install the required libraries using the `pip` package manager.
# - `transformers` for working with pre-trained models from Hugging Face.
# - `timm` (PyTorch Image Models) for accessing a wide range of pre-trained image models.
!pip install transformers timm

# Import the time module to add a delay before restarting the session.
import time

# Import `clear_output` from IPython to clear the notebook output, ensuring a clean display for the user.
from IPython.display import clear_output

# Clear the output after the packages are installed to make the notebook cleaner.
clear_output()

# Print a message to let the user know that the libraries are installed & the session will restart.
print("Necessary Libraries are Installed. Restarting the session!")

# Add a short delay (1 second) before restarting to allow the message to be displayed to the user.
time.sleep(1)

# Import the `os` module to access low-level operating system functionality.
import os

# Use `os._exit(00)` to exit the current Python runtime environment forcefully.
# This effectively simulates a restart in notebook environments like Google Colab or Jupyter.
# After this command, the environment will be restarted & all the packages installed will be properly loaded.
os._exit(00)

In [ ]:
#@title Import Libraries for Depth Estimation with Transformers
'''
This code imports the necessary libraries for performing depth estimation using a pre-trained model from Hugging Face.
The DPT (Dense Prediction Transformer) model is used for depth estimation, which is the task of predicting the distance
of each pixel in an image from the camera. The result is a depth map, which can be used in various computer vision
applications like 3D reconstruction, robotics, & augmented reality.

References:
- Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers/
- DPT Paper: https://arxiv.org/abs/2103.13413
'''

# Import PyTorch (torch), a deep learning library that provides support for tensor operations, building models,
# & performing computations on hardware like CPUs & GPUs. In this script, PyTorch is used to handle model operations.
import torch

# Import the `Image` class from the Python Imaging Library (PIL), specifically from the Pillow module.
# PIL allows easy image loading, conversion, & manipulation (e.g., resizing, format conversion).
# Here, it's used to load images for depth estimation.
from PIL import Image

# Import Matplotlib's `pyplot` module for visualizing images & depth maps.
# `plt` provides functionality for plotting, displaying images, & creating figures with subplots.
import matplotlib.pyplot as plt

# Import NumPy, a powerful library for numerical operations on arrays & matrices.
# NumPy is widely used in image processing to manipulate pixel values & arrays representing images.
import numpy as np

# Import `DPTFeatureExtractor` & `DPTForDepthEstimation` from the Hugging Face `transformers` library.
# `DPTFeatureExtractor` preprocesses images (e.g., resizing, normalization) to make them compatible with the DPT model.
# `DPTForDepthEstimation` is the pre-trained DPT model that performs depth estimation by predicting the depth of each pixel in an image.
from transformers import DPTFeatureExtractor, DPTForDepthEstimation


In [ ]:
#@title Main Run for Depth Estimation using DPT Model
'''
This script demonstrates how to use a pre-trained Dense Prediction Transformer (DPT) model for depth estimation.
Depth estimation is the task of predicting the distance of every pixel in an image from the camera.
This model, which is loaded from the Hugging Face model hub, predicts the depth map for a given input image.

References:
- DPT Paper: https://arxiv.org/abs/2103.13413
- Hugging Face DPT Models: https://huggingface.co/Intel/dpt-large
'''

# The following line installs the required libraries (`transformers` & `timm`) for working with pre-trained models.
# Uncomment & run the line if these libraries are not already installed.
# !pip install transformers timm

# Import Google Drive from Colab to access files stored in the user's Google Drive.
# This is useful in environments like Google Colab where you can store & load images or datasets from Drive.
from google.colab import drive

# Mount Google Drive at the `/content/drive` path to allow access to the user's Drive files in the Colab session.
drive.mount('/content/drive')

# Determine whether a GPU (CUDA) is available for faster computation.
# If CUDA is available, it uses the GPU; otherwise, it defaults to the CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the pre-trained DPT feature extractor & model for depth estimation from Hugging Face.
# `DPTFeatureExtractor` processes images to prepare them for the DPT model by resizing, normalizing, & converting to tensors.
# `DPTForDepthEstimation` is the actual pre-trained model that predicts depth values for each pixel in the input image.
feature_extractor = DPTFeatureExtractor.from_pretrained('Intel/dpt-large')

# Load the DPT model & move it to the selected device (either GPU or CPU).
model = DPTForDepthEstimation.from_pretrained('Intel/dpt-large').to(device)

# Load an image from the user's Google Drive. The image is opened using the PIL library & converted to RGB format.
# The image path needs to point to a valid image in the Google Drive.
image = Image.open("/content/drive/MyDrive/data_535743/coco2017_reduced_1000/val2017/000000453722.jpg").convert("RGB")

# Preprocess the image using the feature extractor to prepare it for the DPT model.
# This step includes resizing & normalizing the image to match the input requirements of the model.
# `return_tensors="pt"` converts the image into a PyTorch tensor. The tensor is then moved to the appropriate device (GPU/CPU).
inputs = feature_extractor(images=image, return_tensors="pt").to(device)

# Perform inference (forward pass) to get the predicted depth values from the model.
# `torch.no_grad()` disables gradient computation, which is not required during inference & saves memory.
with torch.no_grad():
    outputs = model(**inputs)  # The model outputs a depth map prediction.

    # Extract the predicted depth map from the model's output.
    predicted_depth = outputs.predicted_depth

# Remove any additional batch dimension from the depth map using `squeeze()`, & move it back to the CPU.
# Convert the tensor to a NumPy array for further processing.
predicted_depth = predicted_depth.squeeze().cpu().numpy()

# Normalize the depth map for visualization.
# Depth maps are typically floating point values & need to be normalized to the 0-255 range for display.
# Here, we first convert the depth map to an unsigned 8-bit integer using `astype(np.uint8)`.
# The depth map is then resized to match the original image size.
prediction = predicted_depth.astype(np.uint8)

# Convert the normalized depth map into an image using the PIL `Image.fromarray` function.
# The `resize()` method ensures that the depth map has the same dimensions as the original image.
prediction = np.array(Image.fromarray(prediction).resize(image.size))

# Normalize the depth map for visualization between 0 & 1.
# The values are scaled to the [0, 1] range for visualization purposes by subtracting the minimum value & dividing by the range.
prediction = (prediction - prediction.min()) / (prediction.max() - prediction.min())


In [ ]:
#@title Display the Image & the Depth Map for Visualization
'''
This code visualizes the original image alongside the predicted depth map using Matplotlib.
The depth map, which is predicted by a pre-trained model, is displayed with a color map (`inferno`) to provide better visual contrast.
A color bar is also added to indicate the range of depth values, helping to interpret the depth of different areas in the image.

References:
- Matplotlib documentation: https://matplotlib.org/stable/contents.html
'''

# Create a figure with two subplots arranged side by side (1 row, 2 columns).
# The `figsize=(20, 10)` argument specifies that the entire figure should be 20 inches wide & 10 inches tall.
# The first subplot will display the original image, & the second will show the depth map.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Display the original image on the first subplot (`ax1`).
# `imshow(image)` renders the original image (loaded previously using PIL or any other method).
ax1.imshow(image)

# Hide the axis & ticks for a cleaner display of the image.
# `axis('off')` removes the axis lines, ticks, & labels, providing an uncluttered view of the image.
ax1.axis('off')

# Set the title for the first subplot to "Original Image" to clarify that this is the unmodified input image.
ax1.set_title('Original Image')

# Display the depth map on the second subplot (`ax2`).
# The depth map is rendered using `imshow`, & the `cmap='inferno'` argument applies the "inferno" color map, which provides a good contrast for visualizing depth.
# The `aspect='auto'` option ensures that the aspect ratio of the image is maintained, matching the original size of the image.
im = ax2.imshow(prediction, cmap='inferno')  # The depth map, already normalized, is displayed here.

# Hide the axis & ticks for the second subplot as well, making the visualization cleaner.
ax2.axis('off')

# Set the title for the second subplot to "Depth Estimation" to clearly indicate that this represents the depth map.
ax2.set_title('Depth Estimation')

# Add a color bar next to the depth map, showing the scale of depth values.
# `fig.colorbar(im, fraction=0.032, pad=0.04)` adds a color bar to the figure, scaled to match the depth values in the `im` depth map.
# The `fraction` & `pad` arguments control the size & spacing of the color bar relative to the figure.
cbar = fig.colorbar(im, fraction=0.032, pad=0.04)

# Set the label for the color bar to "Depth", indicating that the color gradient corresponds to depth values.
cbar.set_label('Depth')

# Customize the ticks on the color bar to represent depth values from 1 to 0.
# The depth map values are normalized between 0 & 1, so the tick locations are set accordingly (0, 0.25, 0.5, 0.75, 1).
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1])  # Tick positions representing normalized depth values.

# Set the tick labels to show the reverse of the normalized depth (1 to 0).
# This is because in many depth estimation applications, closer objects are represented by larger values.
# The labels will be displayed as [1, 0.75, 0.5, 0.25, 0] to reflect the depth interpretation.
cbar.set_ticklabels([1, 0.75, 0.5, 0.25, 0])

# Display the complete figure with both the original image, depth map, & color bar.
# `plt.show()` renders the figure & shows it in the output, making it interactive in environments like Jupyter notebooks or Colab.
plt.show()

## **8. Mask2Former**

In [ ]:
#@title Install Necessary Libraries & Restart the Session
'''
This block of code installs essential libraries (`transformers` & `timm`) for working with
state-of-the-art models in computer vision & natural language processing. After installing the libraries,
the session is restarted to ensure that the libraries are loaded correctly into the environment.

- `transformers`: A popular library by Hugging Face that provides pre-trained models for NLP tasks & vision-language models.
  Documentation: https://huggingface.co/transformers/

- `timm`: A library by Ross Wightman providing a large collection of pre-trained models for image classification, segmentation, etc.
  Documentation: https://github.com/rwightman/pytorch-image-models

This script is specifically designed for environments like Google Colab or Jupyter Notebooks, where you can dynamically install packages & restart the runtime to ensure proper initialization of installed packages.

References:
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- PyTorch Image Models (timm): https://rwightman.github.io/pytorch-image-models/
'''

# Install the required libraries using the `pip` package manager.
# - `transformers` for working with pre-trained models from Hugging Face.
# - `timm` (PyTorch Image Models) for accessing a wide range of pre-trained image models.
!pip install transformers timm

# Import the time module to add a delay before restarting the session.
import time

# Import `clear_output` from IPython to clear the notebook output, ensuring a clean display for the user.
from IPython.display import clear_output

# Clear the output after the packages are installed to make the notebook cleaner.
clear_output()

# Print a message to let the user know that the libraries are installed & the session will restart.
print("Necessary Libraries are Installed. Restarting the session!")

# Add a short delay (1 second) before restarting to allow the message to be displayed to the user.
time.sleep(1)

# Import the `os` module to access low-level operating system functionality.
import os

# Use `os._exit(00)` to exit the current Python runtime environment forcefully.
# This effectively simulates a restart in notebook environments like Google Colab or Jupyter.
# After this command, the environment will be restarted & all the packages installed will be properly loaded.
os._exit(00)

In [ ]:
#@title Import Necessary Libraries for Universal Segmentation using Mask2Former
'''
This code imports the required libraries to perform universal segmentation using the Mask2Former model.
Mask2Former is a state-of-the-art model designed to handle multiple segmentation tasks, such as instance, panoptic,
and semantic segmentation. The code utilizes Hugging Face's `transformers` library to load the Mask2Former model & its image processor.

References:
- Mask2Former Paper: https://arxiv.org/abs/2112.01527
- Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers/
'''

# Import PyTorch, a deep learning library for tensor computation & model handling.
# PyTorch enables efficient model training & inference on both CPU & GPU.
# In this script, PyTorch is used to handle model operations like inference & transferring data between CPU & GPU.
import torch

# Import the Mask2Former image processor & the segmentation model from Hugging Face's `transformers` library.
# `Mask2FormerImageProcessor`: Prepares images for Mask2Former by applying the necessary transformations like resizing & normalization.
# `Mask2FormerForUniversalSegmentation`: This is the Mask2Former model designed for universal segmentation tasks,
# meaning it can perform semantic, instance, & panoptic segmentation.
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# Import Matplotlib's `pyplot` module for visualizing images, segmentation maps, & other plots.
# `plt` will be used to display the input images & segmentation results.
import matplotlib.pyplot as plt

# Import Matplotlib's `patches` module for creating shapes, such as bounding boxes, for visualizations.
# This is useful for highlighting regions or objects in segmentation tasks by drawing boxes or other shapes around detected objects.
import matplotlib.patches as mpatches

# Import NumPy, a fundamental package for performing numerical operations in Python.
# NumPy is used here to manipulate image arrays & handle data formats required for segmentation outputs.
import numpy as np

# Import `Image` from PIL (Python Imaging Library), a library used for opening, manipulating, & saving images.
# PIL is commonly used for loading images into Python & converting them into formats that models can process.
from PIL import Image

In [ ]:
#@title Main Run: Performing Semantic, Panoptic, & Instance Segmentation using Mask2Former
'''
This script performs semantic, panoptic, & instance segmentation using the Mask2Former model from Hugging Face's `transformers` library.
The script processes an input image & performs inference using pre-trained models to generate segmentation maps for each task.
It then visualizes the results by colorizing the segmentation outputs for easy interpretation.

References:
- Mask2Former Paper: https://arxiv.org/abs/2112.01527
- Hugging Face Documentation for Mask2Former: https://huggingface.co/facebook/mask2former-swin-large-ade-semantic
'''

# Import Google Drive from Colab to access files stored in the user's Google Drive.
# This is useful in environments like Google Colab where you can store & load images or datasets from Drive.
from google.colab import drive

# Mount Google Drive at the `/content/drive` path to allow access to the user's Drive files in the Colab session.
drive.mount('/content/drive')

# Load the input image using PIL (Python Imaging Library) & convert it to RGB format.
# This ensures the image is in the correct format for the Mask2Former model, which expects RGB images.
image = Image.open("/content/drive/MyDrive/data_535743/coco2017_reduced_1000/val2017/000000453722.jpg").convert("RGB")

# Configure the device for computation. If a CUDA-enabled GPU is available, it will be used for faster inference;
# otherwise, the CPU is used.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the Mask2Former model & image processor for semantic segmentation.
# `Mask2FormerImageProcessor` prepares the input image (resizing, normalizing, etc.).
# The pre-trained model "facebook/mask2former-swin-large-ade-semantic" is fine-tuned for semantic segmentation on the ADE20K dataset.
processor_semantic = Mask2FormerImageProcessor.from_pretrained("facebook/mask2former-swin-large-ade-semantic")
model_semantic = Mask2FormerForUniversalSegmentation.from_pretrained(
    "facebook/mask2former-swin-large-ade-semantic"
).to(device)  # Move the model to the appropriate device (GPU/CPU).

# Set the model to evaluation mode, which ensures that certain layers like dropout or batch normalization behave correctly during inference.
model_semantic.eval()

# Prepare the image for the semantic segmentation model using the processor.
# The processor returns a tensor suitable for the model, & it's moved to the appropriate device (GPU/CPU).
inputs_semantic = processor_semantic(images=image, return_tensors="pt").to(device)

# Perform inference for semantic segmentation with no gradient calculation.
# `torch.no_grad()` saves memory & speeds up inference since gradients are not needed.
with torch.no_grad():
    outputs_semantic = model_semantic(**inputs_semantic)

# Post-process the outputs from the semantic segmentation model.
# This step converts the model's raw predictions into human-readable segmentation maps, resizing them to the input image size.
pred_semantic_segmentation = processor_semantic.post_process_semantic_segmentation(
    outputs_semantic, target_sizes=[image.size[::-1]]  # Resize to match the original image's dimensions.
)[0].cpu()  # Move the result back to the CPU for further processing.

# Load the Mask2Former model & image processor for panoptic segmentation.
# The "facebook/mask2former-swin-large-coco-panoptic" model is fine-tuned for panoptic segmentation on the COCO dataset.
processor_panoptic = Mask2FormerImageProcessor.from_pretrained("facebook/mask2former-swin-large-coco-panoptic")
model_panoptic = Mask2FormerForUniversalSegmentation.from_pretrained(
    "facebook/mask2former-swin-large-coco-panoptic"
).to(device)

# Set the panoptic segmentation model to evaluation mode.
model_panoptic.eval()

# Prepare the image for panoptic segmentation using the corresponding processor.
inputs_panoptic = processor_panoptic(images=image, return_tensors="pt").to(device)

# Perform inference for panoptic segmentation.
with torch.no_grad():
    outputs_panoptic = model_panoptic(**inputs_panoptic)

# Post-process the outputs for panoptic segmentation to obtain a human-readable segmentation map & metadata.
pred_panoptic_segmentation = processor_panoptic.post_process_panoptic_segmentation(
    outputs_panoptic, target_sizes=[image.size[::-1]]
)[0]
pred_panoptic_segmentation["segmentation"] = pred_panoptic_segmentation["segmentation"].cpu()  # Move the segmentation map to the CPU.

# Post-process the outputs for instance segmentation (a part of panoptic segmentation).
# The `threshold` & `mask_threshold` are used to filter the instance masks.
pred_instance_segmentation = processor_panoptic.post_process_instance_segmentation(
    outputs_panoptic, threshold=0.5, mask_threshold=0.5, target_sizes=[image.size[::-1]]
)[0]
pred_instance_segmentation["segmentation"] = pred_instance_segmentation["segmentation"].cpu()  # Move the instance segmentation map to the CPU.

# Combine the `id2label` mappings from both the semantic & panoptic models.
# The `id2label` dictionaries map class IDs to human-readable labels (e.g., "person", "car", etc.).
id2label_semantic = model_semantic.config.id2label
id2label_panoptic = model_panoptic.config.id2label

# Create a unified set of labels by combining the labels from both the semantic & panoptic segmentation models.
# Using `set()` ensures that there are no duplicates, & `sorted()` arranges the labels in alphabetical order for consistency.
combined_labels = set(id2label_semantic.values()).union(set(id2label_panoptic.values()))
combined_labels = sorted(list(combined_labels))

# Define a color palette for all potential categories in the segmentation tasks.
# NumPy's `random` function generates random RGB colors for each class label.
# The random seed is set to ensure the colors are consistent across different runs of the script.
np.random.seed(42)
color_palette = {label: np.random.randint(0, 255, 3) for label in combined_labels}

# Visualization: Prepare the semantic segmentation map for display.
semantic_seg = pred_semantic_segmentation.numpy()  # Convert the segmentation map to a NumPy array.
semantic_labels = np.unique(semantic_seg)  # Get the unique class labels in the segmentation map.

# Create a blank image to store the colorized semantic segmentation map.
color_semantic_seg = np.zeros((semantic_seg.shape[0], semantic_seg.shape[1], 3), dtype=np.uint8)

# Loop over each unique label in the semantic segmentation map.
# For each label, assign a color from the color palette & apply it to the corresponding pixels in the colorized map.
for label_id in semantic_labels:
    mask = semantic_seg == label_id  # Create a mask where the pixels correspond to the current label.
    label_name = id2label_semantic.get(label_id, 'Unknown')  # Get the human-readable label for the current class ID.
    color = color_palette.get(label_name, np.random.randint(0, 255, 3))  # Get the color for the label from the palette.
    color_semantic_seg[mask] = color  # Color the pixels corresponding to the label.

# Visualization: Prepare the panoptic segmentation map for display.
panoptic_seg = pred_panoptic_segmentation["segmentation"].numpy()  # Convert the panoptic segmentation map to a NumPy array.
segments_info_panoptic = pred_panoptic_segmentation["segments_info"]  # Get metadata for the panoptic segments.

# Create a blank image to store the colorized panoptic segmentation map.
color_panoptic_seg = np.zeros((panoptic_seg.shape[0], panoptic_seg.shape[1], 3), dtype=np.uint8)

# Loop over each segment in the panoptic segmentation output.
# Each segment has a unique ID & corresponds to a class label (e.g., "person", "car").
for segment in segments_info_panoptic:
    segment_id = segment['id']  # Get the unique segment ID.
    label_id = segment['label_id']  # Get the class label ID for the segment.
    label_name = id2label_panoptic.get(label_id, 'Unknown')  # Get the human-readable label for the class ID.
    mask = panoptic_seg == segment_id  # Create a mask for the pixels corresponding to this segment.
    color = color_palette.get(label_name, np.random.randint(0, 255, 3))  # Get the color for the label from the palette.
    color_panoptic_seg[mask] = color  # Apply the color to the corresponding pixels.

# Visualization: Prepare the instance segmentation map for display.
instance_seg = pred_instance_segmentation["segmentation"].numpy()  # Convert the instance segmentation map to a NumPy array.
segments_info_instance = pred_instance_segmentation["segments_info"]  # Get metadata for the instance segments.

# Create a copy of the original image to visualize the instance segmentation results.
instance_seg_canvas = np.array(image).copy()

# Loop over each instance segment & color the corresponding pixels.
for segment in segments_info_instance:
    segment_id = segment['id']  # Get the unique segment ID.
    label_id = segment['label_id']  # Get the class label ID for the segment.
    label_name = id2label_panoptic.get(label_id, 'Unknown')  # Get the human-readable label for the class ID.
    mask = instance_seg == segment_id  # Create a mask for the pixels corresponding to this instance.
    color = color_palette.get(label_name, np.random.randint(0, 255, 3))  # Get the color for the label from the palette.
    instance_seg_canvas[mask] = color  # Apply the color to the corresponding pixels.

In [ ]:
#@title Display the Results for Semantic, Panoptic, & Instance Segmentation
'''
This section of the code displays the results of semantic, panoptic, & instance segmentation using the Mask2Former model.
It visualizes the original image alongside the segmented outputs, showing how different models break down the image into various segments.
A unified legend is also created to explain the colors used in the segmentations, showing which colors correspond to specific labels.

References:
- Mask2Former Paper: https://arxiv.org/abs/2112.01527
- Matplotlib Documentation: https://matplotlib.org/stable/contents.html
'''

# Create a set to store all unique labels found in the semantic, panoptic, & instance segmentation results.
# This ensures that only the labels present in the current image are used to create the legend.
unique_labels_in_image = set()

# Loop over all unique class labels in the semantic segmentation result.
# For each label, get the corresponding human-readable label name from the `id2label_semantic` dictionary & add it to the set.
for label_id in semantic_labels:
    label_name = id2label_semantic.get(label_id, 'Unknown')  # Fallback to 'Unknown' if the label is not found.
    unique_labels_in_image.add(label_name)

# Loop over the panoptic segmentation results & do the same as above.
# This ensures that any additional labels from the panoptic model are included in the final legend.
for segment in segments_info_panoptic:
    label_id = segment['label_id']
    label_name = id2label_panoptic.get(label_id, 'Unknown')
    unique_labels_in_image.add(label_name)

# Similarly, loop over the instance segmentation results & add the corresponding labels to the set.
# This ensures that the legend includes all labels from instance segmentation as well.
for segment in segments_info_instance:
    label_id = segment['label_id']
    label_name = id2label_panoptic.get(label_id, 'Unknown')
    unique_labels_in_image.add(label_name)

# Create the legend elements using Matplotlib patches (`mpatches.Patch`).
# Each patch represents a class label (e.g., "person", "car") with its corresponding color from the `color_palette`.
# The patches will be displayed as a legend at the bottom of the figure.
legend_elements = [
    mpatches.Patch(facecolor=np.array(color_palette[label]) / 255, label=label) for label in unique_labels_in_image
]

# Create a figure with 4 subplots arranged horizontally using `plt.subplots`.
# `figsize=(20, 10)` sets the size of the entire figure (20 inches wide, 10 inches tall).
fig, axes = plt.subplots(1, 4, figsize=(20, 10))

# Display the original image in the first subplot (`axes[0]`).
# `imshow(image)` renders the original image.
axes[0].imshow(image)

# Turn off the axis for the first subplot to make the visualization cleaner (no axis ticks or labels).
axes[0].axis('off')

# Set the title for the first subplot to "Original Image" with a larger font size for readability.
axes[0].set_title('Original Image', fontsize=20)

# Display the colorized semantic segmentation result in the second subplot (`axes[1]`).
# `color_semantic_seg` is the NumPy array where each pixel has been color-coded based on the segmentation labels.
axes[1].imshow(color_semantic_seg)

# Turn off the axis for the second subplot & set an appropriate title.
axes[1].axis('off')
axes[1].set_title('Semantic Segmentation', fontsize=20)

# Display the colorized panoptic segmentation result in the third subplot (`axes[2]`).
# `color_panoptic_seg` contains the color-coded panoptic segmentation map.
axes[2].imshow(color_panoptic_seg)

# Turn off the axis for the third subplot & set its title to "Panoptic Segmentation".
axes[2].axis('off')
axes[2].set_title('Panoptic Segmentation', fontsize=20)

# Display the colorized instance segmentation result in the fourth subplot (`axes[3]`).
# `instance_seg_canvas` is the copy of the original image where instance segmentation masks have been applied.
axes[3].imshow(instance_seg_canvas)

# Turn off the axis for the fourth subplot & set its title to "Instance Segmentation".
axes[3].axis('off')
axes[3].set_title('Instance Segmentation', fontsize=20)

# Add a global legend at the bottom of the figure to show the correspondence between colors & labels.
# `handles=legend_elements` defines the patches created for each class label.
# `loc='lower center'` positions the legend at the bottom, & `bbox_to_anchor` fine-tunes the location.
# `ncol=10` ensures the legend is arranged in 10 columns for a compact layout, & `fontsize=14` sets the font size.
fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.2), ncol=10, fontsize=14)

# Adjust the layout of the subplots to make sure they don't overlap & are well spaced.
plt.tight_layout()

# Display the final figure with all subplots & the legend.
plt.show()

In [ ]:
#@title Modify the Locale Configuration for UTF-8 Encoding
'''
This code snippet modifies the default locale configuration in Python to ensure that UTF-8 encoding is used.
In some environments (e.g., Jupyter notebooks, Google Colab, or certain operating systems), the default locale
may not support UTF-8, which can lead to encoding issues when handling non-ASCII text (e.g., characters with accents,
symbols from different languages).

This modification can be helpful when working with text data in different languages or when reading/writing files
containing non-ASCII characters.

References:
- Python Locale Module Documentation: https://docs.python.org/3/library/locale.html
'''

# Import the `locale` module, which provides access to the underlying locale settings in Python.
# Locale settings affect how programs handle tasks such as string encoding, sorting, & formatting dates & numbers.
import locale

# Modify the `locale.getpreferredencoding` function to return "UTF-8" by default.
# This ensures that when Python retrieves the system's preferred encoding (usually through `locale.getpreferredencoding()`),
# it will always use UTF-8, which is a widely-used standard for encoding Unicode characters.
# Lambda function returns "UTF-8" to ensure that all text input/output operations are handled with this encoding.
locale.getpreferredencoding = lambda: "UTF-8"

# <font color="#418FDE" size="6.5" uppercase>**B: Intermediate Segmentation & Detection Experiments**</font>
----

In this lecture, you learned to:

* Develop transferred-learned & fine-tuned models for various segmentation tasks
* Apply pre-trained models for various segmentation tasks

In the next Module (Module 11), we will go over Natural Language Processing, Part 1.